# Notebook 37 — Mechanism and parameterization sensitivity matrix

## Goal

Build a mechanism / parameterization sensitivity matrix before writing the non-geometric-acceleration claim review.

The purpose is to test whether additional PyBaMM-supported physics branches or diagnostic parameterization variants can produce stable, positive, engineering-significant non-geometric terminal-Q acceleration, or whether they mainly affect voltage-boundary behavior, microstate stress, and plating-margin admissibility.

Core question:

> After adding one missing-physics / parameterization branch at a time, does any branch create stable positive `Δt_resid(Q)`, or does the DC–AC timing remain geometry-dominated?

## Scientific context

Day30–Day36 established the current project boundary:

- raw DC–AC timing gain is mostly prescribed-current geometry;
- `Δt_resid(Q)` remains near zero under tested DFN/SPMe/SPM and parameter-set branches;
- DC–AC creates real microstate and plating-margin effects;
- state-aware scheduling improves hard U_NE margin;
- differential double-layer surface form does not explain the absence of engineering-significant positive non-geometric acceleration.

However, before writing a final claim review, two additional missing-physics / parameterization directions must be audited:

1. **Electrolyte transport parameterization sensitivity**
   - default electrolyte diffusivity / conductivity;
   - flattened transport functions;
   - amplified concentration-dependent transport nonlinearity.

2. **OCP / hysteresis / path-dependence support**
   - audit whether the current PyBaMM / Chen2020 DFN configuration supports a relevant hysteresis or dynamic-OCP branch;
   - if unsupported, record as deferred rather than forcing a pseudo-model.

## Protocol set

Every runnable branch should use the same minimal protocol set:

- DC `0.3C`
- fixed DC–AC `0.3C + 0.38C`
- fixed DC–AC `0.3C + 0.7C`
- scheduled `sched_v2_conservative`

The fixed `AC0.7` stress control is mandatory because it tests whether a branch changes high-amplitude risk / residual behavior.

## Day37A scope

Day37A is only a capability and branch-registry audit. It should not yet run the full matrix.

Day37A outputs:

- `data/day37_metadata.json`
- `data/day37_parameter_function_audit.csv`
- `data/day37_option_capability_audit.csv`
- `data/day37_transport_branch_registry.csv`
- `data/day37_mechanism_branch_registry.csv`
- `data/day37_protocol_table_preview.csv`

## Interpretation boundary

- Day37 is a sensitivity matrix, not a truth model.
- Artificial flattened / amplified transport branches are diagnostic perturbations, not physical claims.
- Unsupported hysteresis / dynamic-OCP branches must be recorded as deferred.
- A combined branch is a stress-test branch, not a validated model.
- The claim review should only be written after Day37 results are complete.

In [1]:
# Cell 1 — Imports, paths, and Day37 metadata（导入库、路径与Day37元数据）

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pybamm

warnings.filterwarnings("default")

REPO = Path("/Users/louislu/pybamm-dcac-superimposed")
DATA_DIR = REPO / "data"
FIG_DIR = REPO / "figures"
DOCS_DIR = REPO / "docs"

for p in [DATA_DIR, FIG_DIR, DOCS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PRIMARY_PARAMETER_SET = "Chen2020"
MODEL_TYPE = "DFN"
FREQUENCY_MODE = "set_rebased"

DC_C = 0.3
TAU_FACTOR_MAIN = 1.5
TAU_REF_SET_S = 36.316532231592305
F_MAIN_HZ = 1.0 / (2.0 * np.pi * TAU_FACTOR_MAIN * TAU_REF_SET_S)
PERIOD_MAIN_S = 1.0 / F_MAIN_HZ

PLATING_MARGIN_SAFE_BUFFER_V = 0.050

DAY37_METADATA = {
    "notebook": "37_mechanism_parameterization_sensitivity_matrix.ipynb",
    "parameter_set": PRIMARY_PARAMETER_SET,
    "model_type": MODEL_TYPE,
    "frequency_mode": FREQUENCY_MODE,
    "tau_factor_main": TAU_FACTOR_MAIN,
    "tau_ref_set_s": TAU_REF_SET_S,
    "f_main_Hz": F_MAIN_HZ,
    "purpose": "mechanism and parameterization sensitivity matrix before non-geometric acceleration claim review",
    "required_protocols": ["DC03", "fixed_AC0p38", "fixed_AC0p7", "sched_v2_conservative"],
    "lineage": "Day37 follows Day30-Day36; Day36 closed double-layer surface-form sensitivity.",
}

metadata_path = DATA_DIR / "day37_metadata.json"
metadata_path.write_text(json.dumps(DAY37_METADATA, indent=2), encoding="utf-8")

print("[OK] repo:", REPO)
print("[OK] pybamm version:", pybamm.__version__)
print("[OK] f_main_Hz:", F_MAIN_HZ)
print("[OK] period_main_s:", PERIOD_MAIN_S)
print("[OK] saved metadata:", metadata_path)

[OK] repo: /Users/louislu/pybamm-dcac-superimposed
[OK] pybamm version: 26.3.1
[OK] f_main_Hz: 0.002921625190367048
[OK] period_main_s: 342.27525258788194
[OK] saved metadata: /Users/louislu/pybamm-dcac-superimposed/data/day37_metadata.json


In [2]:
# Cell 2 — Audit Chen2020 transport and OCP parameter functions（审计Chen2020输运与OCP参数函数）

parameter_values = pybamm.ParameterValues(PRIMARY_PARAMETER_SET)
param_keys = sorted(list(parameter_values.keys()))

PARAMETER_AUDIT_KEYS = [
    "Electrolyte diffusivity [m2.s-1]",
    "Electrolyte conductivity [S.m-1]",
    "Negative electrode OCP [V]",
    "Positive electrode OCP [V]",
    "Negative electrode exchange-current density [A.m-2]",
    "Positive electrode exchange-current density [A.m-2]",
    "Negative electrode double-layer capacity [F.m-2]",
    "Positive electrode double-layer capacity [F.m-2]",
]

parameter_rows = []

for key in PARAMETER_AUDIT_KEYS:
    available = key in parameter_values
    value_type = ""
    value_repr = ""
    callable_flag = False

    if available:
        try:
            value = parameter_values[key]
            value_type = type(value).__name__
            value_repr = repr(value)
            callable_flag = callable(value)
        except Exception as e:
            value_type = "read_error"
            value_repr = repr(e)
            callable_flag = False

    parameter_rows.append({
        "parameter_key": key,
        "available": available,
        "value_type": value_type,
        "callable": callable_flag,
        "value_repr": value_repr[:500],
    })

parameter_function_audit_df = pd.DataFrame(parameter_rows)

out = DATA_DIR / "day37_parameter_function_audit.csv"
parameter_function_audit_df.to_csv(out, index=False)

print("[OK] saved parameter function audit:", out)
display(parameter_function_audit_df)

# Broader inventory for transport / OCP / hysteresis-ish names.
keyword_terms = ["electrolyte", "diffus", "conduct", "ocp", "open-circuit", "hyster", "entropy"]
keyword_rows = []

for key in param_keys:
    low = key.lower()
    for term in keyword_terms:
        if term in low:
            keyword_rows.append({"search_term": term, "parameter_key": key})

keyword_inventory_df = pd.DataFrame(keyword_rows).drop_duplicates()

out = DATA_DIR / "day37_parameter_keyword_inventory.csv"
keyword_inventory_df.to_csv(out, index=False)

print("[OK] saved parameter keyword inventory:", out)
display(keyword_inventory_df.head(200))

[OK] saved parameter function audit: /Users/louislu/pybamm-dcac-superimposed/data/day37_parameter_function_audit.csv


,parameter_key,available,value_type,callable,value_repr
0,Electrolyte diffusivity [m2.s-1],True,function,True,<function electrolyte_diffusivity_Nyman2008 at...
1,Electrolyte conductivity [S.m-1],True,function,True,<function electrolyte_conductivity_Nyman2008 a...
2,Negative electrode OCP [V],True,function,True,<function graphite_LGM50_ocp_Chen2020 at 0x116...
3,Positive electrode OCP [V],True,function,True,<function nmc_LGM50_ocp_Chen2020 at 0x116e372e0>
4,Negative electrode exchange-current density [A...,True,function,True,<function graphite_LGM50_electrolyte_exchange_...
5,Positive electrode exchange-current density [A...,True,function,True,<function nmc_LGM50_electrolyte_exchange_curre...
6,Negative electrode double-layer capacity [F.m-2],True,float,False,0.2
7,Positive electrode double-layer capacity [F.m-2],True,float,False,0.2


[OK] saved parameter keyword inventory: /Users/louislu/pybamm-dcac-superimposed/data/day37_parameter_keyword_inventory.csv


,search_term,parameter_key
0,diffus,EC diffusivity [m2.s-1]
1,electrolyte,EC initial concentration in electrolyte [mol.m-3]
2,electrolyte,Electrolyte conductivity [S.m-1]
3,conduct,Electrolyte conductivity [S.m-1]
4,electrolyte,Electrolyte diffusivity [m2.s-1]
5,diffus,Electrolyte diffusivity [m2.s-1]
6,electrolyte,Initial concentration in electrolyte [mol.m-3]
7,conduct,Negative current collector conductivity [S.m-1]
8,conduct,Negative current collector thermal conductivit...
9,electrolyte,Negative electrode Bruggeman coefficient (elec...


In [3]:
# Cell 3 — Model option capability audit（模型选项可用性审计）

# Candidate option branches to test. Some may not be valid for Chen2020 / DFN.
OPTION_CANDIDATES = [
    {
        "branch_id": "baseline_false",
        "options": {"thermal": "isothermal", "surface form": "false"},
        "purpose": "baseline prior mainline",
    },
    {
        "branch_id": "double_layer_differential",
        "options": {"thermal": "isothermal", "surface form": "differential"},
        "purpose": "Day36 double-layer dynamic branch",
    },
    {
        "branch_id": "surface_form_algebraic",
        "options": {"thermal": "isothermal", "surface form": "algebraic"},
        "purpose": "Day36 algebraic diagnostic",
    },
    {
        "branch_id": "particle_mechanics_swelling_only",
        "options": {"thermal": "isothermal", "particle mechanics": "swelling only"},
        "purpose": "capability audit only; not necessarily Day37 main branch",
    },
    {
        "branch_id": "particle_mechanics_swelling_cracking",
        "options": {"thermal": "isothermal", "particle mechanics": "swelling and cracking"},
        "purpose": "capability audit only; not necessarily Day37 main branch",
    },
    {
        "branch_id": "sei_solvent_diffusion_limited",
        "options": {"thermal": "isothermal", "SEI": "solvent-diffusion limited"},
        "purpose": "capability audit only; side reaction branch",
    },
    {
        "branch_id": "lithium_plating_reversible",
        "options": {"thermal": "isothermal", "lithium plating": "reversible"},
        "purpose": "capability audit only; not used as terminal-Q acceleration branch unless cleanly supported",
    },
]

option_rows = []
option_variable_rows = []

for cand in OPTION_CANDIDATES:
    branch_id = cand["branch_id"]
    options = cand["options"]
    print("[BUILD]", branch_id, options)

    try:
        model = pybamm.lithium_ion.DFN(options=options)
        sim = pybamm.Simulation(model, parameter_values=parameter_values)
        sim.build()
        keys = sorted(list(sim.built_model.variables.keys()))

        option_rows.append({
            "branch_id": branch_id,
            "options_json": json.dumps(options),
            "build_status": "ok",
            "n_variables": len(keys),
            "purpose": cand["purpose"],
            "error": "",
        })

        for term in ["hyster", "plating", "sei", "mechanics", "crack", "stress", "surface potential difference", "overpotential"]:
            matches = [k for k in keys if term.lower() in k.lower()]
            for k in matches:
                option_variable_rows.append({
                    "branch_id": branch_id,
                    "search_term": term,
                    "variable_key": k,
                })

        print("[OK]", branch_id, "n_variables:", len(keys))

    except Exception as e:
        option_rows.append({
            "branch_id": branch_id,
            "options_json": json.dumps(options),
            "build_status": "failed",
            "n_variables": np.nan,
            "purpose": cand["purpose"],
            "error": repr(e),
        })
        print("[FAILED]", branch_id, repr(e))

option_capability_audit_df = pd.DataFrame(option_rows)
option_variable_inventory_df = pd.DataFrame(option_variable_rows).drop_duplicates()

out = DATA_DIR / "day37_option_capability_audit.csv"
option_capability_audit_df.to_csv(out, index=False)

out2 = DATA_DIR / "day37_option_variable_inventory.csv"
option_variable_inventory_df.to_csv(out2, index=False)

print("[OK] saved option capability audit:", out)
print("[OK] saved option variable inventory:", out2)

display(option_capability_audit_df)
display(option_variable_inventory_df.head(200))

[BUILD] baseline_false {'thermal': 'isothermal', 'surface form': 'false'}
[OK] baseline_false n_variables: 515
[BUILD] double_layer_differential {'thermal': 'isothermal', 'surface form': 'differential'}
[OK] double_layer_differential n_variables: 522
[BUILD] surface_form_algebraic {'thermal': 'isothermal', 'surface form': 'algebraic'}
[OK] surface_form_algebraic n_variables: 522
[BUILD] particle_mechanics_swelling_only {'thermal': 'isothermal', 'particle mechanics': 'swelling only'}
[FAILED] particle_mechanics_swelling_only KeyError("Parameter 'Negative electrode partial molar volume [m3.mol-1]' not found. 'Negative electrode partial molar volume [m3.mol-1]' not found. Best matches are ['SEI partial molar volume [m3.mol-1]', 'Negative electrode reaction-driven LAM factor [m3.mol-1]', 'Negative electrode active material volume fraction']")
[BUILD] particle_mechanics_swelling_cracking {'thermal': 'isothermal', 'particle mechanics': 'swelling and cracking'}
[FAILED] particle_mechanics_swe

,branch_id,options_json,build_status,n_variables,purpose,error
0,baseline_false,"{""thermal"": ""isothermal"", ""surface form"": ""fal...",ok,515.0,baseline prior mainline,
1,double_layer_differential,"{""thermal"": ""isothermal"", ""surface form"": ""dif...",ok,522.0,Day36 double-layer dynamic branch,
2,surface_form_algebraic,"{""thermal"": ""isothermal"", ""surface form"": ""alg...",ok,522.0,Day36 algebraic diagnostic,
3,particle_mechanics_swelling_only,"{""thermal"": ""isothermal"", ""particle mechanics""...",failed,NaN,capability audit only; not necessarily Day37 m...,"KeyError(""Parameter 'Negative electrode partia..."
4,particle_mechanics_swelling_cracking,"{""thermal"": ""isothermal"", ""particle mechanics""...",failed,NaN,capability audit only; not necessarily Day37 m...,"KeyError(""Parameter 'Negative electrode initia..."
5,sei_solvent_diffusion_limited,"{""thermal"": ""isothermal"", ""SEI"": [""solvent-dif...",ok,519.0,capability audit only; side reaction branch,
6,lithium_plating_reversible,"{""thermal"": ""isothermal"", ""lithium plating"": [...",failed,NaN,capability audit only; not used as terminal-Q ...,"KeyError(""Parameter 'Exchange-current density ..."


,branch_id,search_term,variable_key
0,baseline_false,plating,Loss of capacity to negative lithium plating [...
1,baseline_false,plating,Loss of capacity to positive lithium plating [...
2,baseline_false,plating,Loss of lithium to negative lithium plating [mol]
3,baseline_false,plating,Loss of lithium to positive lithium plating [mol]
4,baseline_false,plating,Negative electrode lithium plating interfacial...
...,...,...,...
195,double_layer_differential,sei,Positive SEI on cracks thickness [m]
196,double_layer_differential,sei,Positive SEI thickness [m]
197,double_layer_differential,sei,Positive electrode SEI film overpotential [V]
198,double_layer_differential,sei,Positive electrode SEI interfacial current den...


In [4]:
# Cell 4 — Transport branch registry design（输运参数化分支注册表设计）

# Day37 transport branches are diagnostic perturbations.
# They are not claimed to be physically validated transport models.

transport_branch_rows = [
    {
        "branch_id": "baseline_default_transport",
        "branch_type": "transport",
        "surface_form": "false",
        "transport_modification": "default",
        "De_modification": "default",
        "kappa_modification": "default",
        "runnable_status": "planned",
        "physical_status": "baseline",
        "interpretation_boundary": "Chen2020 default electrolyte transport parameterization.",
    },
    {
        "branch_id": "transport_flattened",
        "branch_type": "transport",
        "surface_form": "false",
        "transport_modification": "flattened",
        "De_modification": "replace electrolyte diffusivity with local reference constant if callable",
        "kappa_modification": "replace electrolyte conductivity with local reference constant if callable",
        "runnable_status": "planned_pending_implementation",
        "physical_status": "diagnostic_not_truth_model",
        "interpretation_boundary": "Tests whether concentration dependence of electrolyte transport matters; not a physical claim.",
    },
    {
        "branch_id": "transport_nonlinearity_amplified",
        "branch_type": "transport",
        "surface_form": "false",
        "transport_modification": "nonlinearity_amplified",
        "De_modification": "diagnostic amplified concentration dependence around reference concentration",
        "kappa_modification": "diagnostic amplified concentration dependence around reference concentration",
        "runnable_status": "planned_pending_implementation",
        "physical_status": "diagnostic_stress_test_not_truth_model",
        "interpretation_boundary": "Stress-tests stronger transport nonlinearity; not a validated electrolyte model.",
    },
]

transport_branch_registry_df = pd.DataFrame(transport_branch_rows)

out = DATA_DIR / "day37_transport_branch_registry.csv"
transport_branch_registry_df.to_csv(out, index=False)

print("[OK] saved transport branch registry:", out)
display(transport_branch_registry_df)


[OK] saved transport branch registry: /Users/louislu/pybamm-dcac-superimposed/data/day37_transport_branch_registry.csv


,branch_id,branch_type,surface_form,transport_modification,De_modification,kappa_modification,runnable_status,physical_status,interpretation_boundary
0,baseline_default_transport,transport,false,default,default,default,planned,baseline,Chen2020 default electrolyte transport paramet...
1,transport_flattened,transport,false,flattened,replace electrolyte diffusivity with local ref...,replace electrolyte conductivity with local re...,planned_pending_implementation,diagnostic_not_truth_model,Tests whether concentration dependence of elec...
2,transport_nonlinearity_amplified,transport,false,nonlinearity_amplified,diagnostic amplified concentration dependence ...,diagnostic amplified concentration dependence ...,planned_pending_implementation,diagnostic_stress_test_not_truth_model,Stress-tests stronger transport nonlinearity; ...


In [5]:
# Cell 5 — Mechanism branch registry and protocol preview（机制分支注册表与协议预览）

# Determine support from option audit.
def option_status(branch_id):
    rows = option_capability_audit_df[option_capability_audit_df["branch_id"] == branch_id]
    if len(rows) == 0:
        return "not_tested"
    return rows["build_status"].iloc[0]


mechanism_branch_rows = [
    {
        "branch_id": "baseline_default",
        "branch_type": "baseline",
        "options": {"thermal": "isothermal", "surface form": "false"},
        "transport_branch_id": "baseline_default_transport",
        "runnable_status": "planned",
        "physical_status": "baseline",
        "interpretation_boundary": "Reference branch for Day37.",
    },
    {
        "branch_id": "double_layer_differential",
        "branch_type": "double_layer",
        "options": {"thermal": "isothermal", "surface form": "differential"},
        "transport_branch_id": "baseline_default_transport",
        "runnable_status": option_status("double_layer_differential"),
        "physical_status": "pybamm_supported",
        "interpretation_boundary": "Day36-supported branch; used to keep double-layer sensitivity in matrix.",
    },
    {
        "branch_id": "transport_flattened",
        "branch_type": "transport_parameterization",
        "options": {"thermal": "isothermal", "surface form": "false"},
        "transport_branch_id": "transport_flattened",
        "runnable_status": "planned_pending_implementation",
        "physical_status": "diagnostic_not_truth_model",
        "interpretation_boundary": "One-factor transport sensitivity branch.",
    },
    {
        "branch_id": "transport_nonlinearity_amplified",
        "branch_type": "transport_parameterization",
        "options": {"thermal": "isothermal", "surface form": "false"},
        "transport_branch_id": "transport_nonlinearity_amplified",
        "runnable_status": "planned_pending_implementation",
        "physical_status": "diagnostic_stress_test_not_truth_model",
        "interpretation_boundary": "One-factor stress test for stronger concentration-dependent transport.",
    },
    {
        "branch_id": "ocp_hysteresis_dynamic",
        "branch_type": "ocp_hysteresis",
        "options": {"thermal": "isothermal"},
        "transport_branch_id": "baseline_default_transport",
        "runnable_status": "deferred_pending_supported_option_identification",
        "physical_status": "not_yet_supported_in_current_audit",
        "interpretation_boundary": "No branch should be run until a valid PyBaMM option / model configuration is identified.",
    },
    {
        "branch_id": "combined_sensitivity",
        "branch_type": "combined",
        "options": {"thermal": "isothermal", "surface form": "differential"},
        "transport_branch_id": "transport_nonlinearity_amplified",
        "runnable_status": "planned_after_individual_branches",
        "physical_status": "diagnostic_stress_test_not_truth_model",
        "interpretation_boundary": "Combined perturbation only after one-factor branches are audited.",
    },
]

mechanism_branch_registry_df = pd.DataFrame(mechanism_branch_rows)

out = DATA_DIR / "day37_mechanism_branch_registry.csv"
mechanism_branch_registry_df.to_csv(out, index=False)

print("[OK] saved mechanism branch registry:", out)
display(mechanism_branch_registry_df)


# Protocol preview. fixed AC0.7 is mandatory.
base_protocol_specs = [
    {
        "protocol_id": "DC03",
        "protocol_type": "DC",
        "case_role": "dc_reference",
        "dc_c": DC_C,
        "fixed_ac_c": 0.0,
        "early_ac_c": 0.0,
        "mid_ac_c": 0.0,
        "late_ac_c": 0.0,
        "schedule_type": "constant",
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
        "required": True,
    },
    {
        "protocol_id": "fixed_AC0p38",
        "protocol_type": "DCAC",
        "case_role": "near_boundary_baseline",
        "dc_c": DC_C,
        "fixed_ac_c": 0.38,
        "early_ac_c": 0.38,
        "mid_ac_c": 0.38,
        "late_ac_c": 0.38,
        "schedule_type": "constant",
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
        "required": True,
    },
    {
        "protocol_id": "fixed_AC0p7",
        "protocol_type": "DCAC",
        "case_role": "high_amplitude_stress_control",
        "dc_c": DC_C,
        "fixed_ac_c": 0.70,
        "early_ac_c": 0.70,
        "mid_ac_c": 0.70,
        "late_ac_c": 0.70,
        "schedule_type": "constant",
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
        "required": True,
    },
    {
        "protocol_id": "sched_v2_conservative",
        "protocol_type": "DCAC_SCHEDULED",
        "case_role": "day35_primary_candidate",
        "dc_c": DC_C,
        "fixed_ac_c": np.nan,
        "early_ac_c": 0.38,
        "mid_ac_c": 0.10,
        "late_ac_c": 0.00,
        "schedule_type": "qfrac_time_proxy",
        "qfrac_derate": 0.70,
        "qfrac_ac_off": 0.80,
        "required": True,
    },
]

protocol_preview_rows = []

for _, branch in mechanism_branch_registry_df.iterrows():
    for proto in base_protocol_specs:
        protocol_preview_rows.append({
            "branch_id": branch["branch_id"],
            "branch_type": branch["branch_type"],
            "transport_branch_id": branch["transport_branch_id"],
            "runnable_status": branch["runnable_status"],
            "protocol_id": proto["protocol_id"],
            "protocol_type": proto["protocol_type"],
            "case_role": proto["case_role"],
            "required_protocol": proto["required"],
            "dc_c": proto["dc_c"],
            "fixed_ac_c": proto["fixed_ac_c"],
            "early_ac_c": proto["early_ac_c"],
            "mid_ac_c": proto["mid_ac_c"],
            "late_ac_c": proto["late_ac_c"],
            "schedule_type": proto["schedule_type"],
            "qfrac_derate": proto["qfrac_derate"],
            "qfrac_ac_off": proto["qfrac_ac_off"],
        })

protocol_preview_df = pd.DataFrame(protocol_preview_rows)

out = DATA_DIR / "day37_protocol_table_preview.csv"
protocol_preview_df.to_csv(out, index=False)

print("[OK] saved protocol table preview:", out)
display(protocol_preview_df)

[OK] saved mechanism branch registry: /Users/louislu/pybamm-dcac-superimposed/data/day37_mechanism_branch_registry.csv


,branch_id,branch_type,options,transport_branch_id,runnable_status,physical_status,interpretation_boundary
0,baseline_default,baseline,"{'thermal': 'isothermal', 'surface form': 'fal...",baseline_default_transport,planned,baseline,Reference branch for Day37.
1,double_layer_differential,double_layer,"{'thermal': 'isothermal', 'surface form': 'dif...",baseline_default_transport,ok,pybamm_supported,Day36-supported branch; used to keep double-la...
2,transport_flattened,transport_parameterization,"{'thermal': 'isothermal', 'surface form': 'fal...",transport_flattened,planned_pending_implementation,diagnostic_not_truth_model,One-factor transport sensitivity branch.
3,transport_nonlinearity_amplified,transport_parameterization,"{'thermal': 'isothermal', 'surface form': 'fal...",transport_nonlinearity_amplified,planned_pending_implementation,diagnostic_stress_test_not_truth_model,One-factor stress test for stronger concentrat...
4,ocp_hysteresis_dynamic,ocp_hysteresis,{'thermal': 'isothermal'},baseline_default_transport,deferred_pending_supported_option_identification,not_yet_supported_in_current_audit,No branch should be run until a valid PyBaMM o...
5,combined_sensitivity,combined,"{'thermal': 'isothermal', 'surface form': 'dif...",transport_nonlinearity_amplified,planned_after_individual_branches,diagnostic_stress_test_not_truth_model,Combined perturbation only after one-factor br...


[OK] saved protocol table preview: /Users/louislu/pybamm-dcac-superimposed/data/day37_protocol_table_preview.csv


,branch_id,branch_type,transport_branch_id,runnable_status,protocol_id,protocol_type,case_role,required_protocol,dc_c,fixed_ac_c,early_ac_c,mid_ac_c,late_ac_c,schedule_type,qfrac_derate,qfrac_ac_off
0,baseline_default,baseline,baseline_default_transport,planned,DC03,DC,dc_reference,True,0.3,0.00,0.00,0.00,0.00,constant,NaN,NaN
1,baseline_default,baseline,baseline_default_transport,planned,fixed_AC0p38,DCAC,near_boundary_baseline,True,0.3,0.38,0.38,0.38,0.38,constant,NaN,NaN
2,baseline_default,baseline,baseline_default_transport,planned,fixed_AC0p7,DCAC,high_amplitude_stress_control,True,0.3,0.70,0.70,0.70,0.70,constant,NaN,NaN
3,baseline_default,baseline,baseline_default_transport,planned,sched_v2_conservative,DCAC_SCHEDULED,day35_primary_candidate,True,0.3,NaN,0.38,0.10,0.00,qfrac_time_proxy,0.7,0.8
4,double_layer_differential,double_layer,baseline_default_transport,ok,DC03,DC,dc_reference,True,0.3,0.00,0.00,0.00,0.00,constant,NaN,NaN
5,double_layer_differential,double_layer,baseline_default_transport,ok,fixed_AC0p38,DCAC,near_boundary_baseline,True,0.3,0.38,0.38,0.38,0.38,constant,NaN,NaN
6,double_layer_differential,double_layer,baseline_default_transport,ok,fixed_AC0p7,DCAC,high_amplitude_stress_control,True,0.3,0.70,0.70,0.70,0.70,constant,NaN,NaN
7,double_layer_differential,double_layer,baseline_default_transport,ok,sched_v2_conservative,DCAC_SCHEDULED,day35_primary_candidate,True,0.3,NaN,0.38,0.10,0.00,qfrac_time_proxy,0.7,0.8
8,transport_flattened,transport_parameterization,transport_flattened,planned_pending_implementation,DC03,DC,dc_reference,True,0.3,0.00,0.00,0.00,0.00,constant,NaN,NaN
9,transport_flattened,transport_parameterization,transport_flattened,planned_pending_implementation,fixed_AC0p38,DCAC,near_boundary_baseline,True,0.3,0.38,0.38,0.38,0.38,constant,NaN,NaN


## Day37B1 — Branch implementation and runnable matrix freeze

Day37A showed:

- Chen2020 electrolyte diffusivity and conductivity are callable functions.
- OCP functions are callable, but no supported dynamic-OCP / hysteresis branch has been identified in the current audit.
- `surface form = differential` is supported from Day36.
- SEI builds, but it is a degradation side-reaction branch and is not included in the clean Day37 non-geometric acceleration matrix.
- particle mechanics and reversible lithium-plating branches failed due missing parameters in the current Chen2020 configuration.

Day37B1 now implements the runnable matrix:

1. `baseline_default`
2. `double_layer_differential`
3. `transport_flattened`
4. `transport_nonlinearity_amplified`
5. `combined_sensitivity`

The mandatory protocol set is:

- `DC03`
- `fixed_AC0p38`
- `fixed_AC0p7`
- `sched_v2_conservative`

`fixed_AC0p7` is retained as the high-amplitude stress control.

In [6]:
# Cell 6 — Define diagnostic transport parameterization functions（定义诊断性输运参数化函数）

# Original Chen2020 electrolyte transport functions.
ORIG_DE_FUNC = parameter_values["Electrolyte diffusivity [m2.s-1]"]
ORIG_KAPPA_FUNC = parameter_values["Electrolyte conductivity [S.m-1]"]

C_E_REF_MOLM3 = 1000.0
T_REF_K = 298.15
TRANSPORT_NONLINEARITY_POWER = 1.5

def _to_float_value(x):
    """Convert PyBaMM scalar / numpy scalar / float into Python float."""
    try:
        if hasattr(x, "evaluate"):
            return float(np.asarray(x.evaluate()).reshape(-1)[0])
        return float(np.asarray(x).reshape(-1)[0])
    except Exception:
        return float(x)

# Reference constants for flattened transport.
DE_REF = _to_float_value(ORIG_DE_FUNC(C_E_REF_MOLM3, T_REF_K))
KAPPA_REF = _to_float_value(ORIG_KAPPA_FUNC(C_E_REF_MOLM3, T_REF_K))

def electrolyte_diffusivity_flattened(c_e, T):
    """
    Diagnostic flattened electrolyte diffusivity.
    Not a physical truth model.
    """
    return DE_REF

def electrolyte_conductivity_flattened(c_e, T):
    """
    Diagnostic flattened electrolyte conductivity.
    Not a physical truth model.
    """
    return KAPPA_REF

def electrolyte_diffusivity_nonlinearity_amplified(c_e, T):
    """
    Diagnostic amplified concentration-dependence branch.
    Keeps the original Chen2020 function but amplifies concentration dependence around C_E_REF.
    Not a validated electrolyte model.
    """
    return ORIG_DE_FUNC(c_e, T) * (c_e / C_E_REF_MOLM3) ** TRANSPORT_NONLINEARITY_POWER

def electrolyte_conductivity_nonlinearity_amplified(c_e, T):
    """
    Diagnostic amplified concentration-dependence branch.
    Keeps the original Chen2020 function but amplifies concentration dependence around C_E_REF.
    Not a validated electrolyte model.
    """
    return ORIG_KAPPA_FUNC(c_e, T) * (c_e / C_E_REF_MOLM3) ** TRANSPORT_NONLINEARITY_POWER

transport_function_rows = [
    {
        "transport_branch_id": "baseline_default_transport",
        "De_function": "Chen2020 default",
        "kappa_function": "Chen2020 default",
        "De_ref_at_1000_298K": DE_REF,
        "kappa_ref_at_1000_298K": KAPPA_REF,
        "interpretation": "baseline",
    },
    {
        "transport_branch_id": "transport_flattened",
        "De_function": "constant DE_REF",
        "kappa_function": "constant KAPPA_REF",
        "De_ref_at_1000_298K": DE_REF,
        "kappa_ref_at_1000_298K": KAPPA_REF,
        "interpretation": "diagnostic flattening of concentration dependence",
    },
    {
        "transport_branch_id": "transport_nonlinearity_amplified",
        "De_function": "default * (c_e/C_REF)^power",
        "kappa_function": "default * (c_e/C_REF)^power",
        "De_ref_at_1000_298K": DE_REF,
        "kappa_ref_at_1000_298K": KAPPA_REF,
        "interpretation": f"diagnostic amplified concentration dependence, power={TRANSPORT_NONLINEARITY_POWER}",
    },
]

transport_function_audit_df = pd.DataFrame(transport_function_rows)

out = DATA_DIR / "day37_transport_function_implementation_audit.csv"
transport_function_audit_df.to_csv(out, index=False)

print("[OK] De_ref:", DE_REF)
print("[OK] kappa_ref:", KAPPA_REF)
print("[OK] saved transport function implementation audit:", out)
display(transport_function_audit_df)


[OK] De_ref: 1.7694000000000006e-10
[OK] kappa_ref: 0.9487000000000005
[OK] saved transport function implementation audit: /Users/louislu/pybamm-dcac-superimposed/data/day37_transport_function_implementation_audit.csv


,transport_branch_id,De_function,kappa_function,De_ref_at_1000_298K,kappa_ref_at_1000_298K,interpretation
0,baseline_default_transport,Chen2020 default,Chen2020 default,1.769400e-10,0.9487,baseline
1,transport_flattened,constant DE_REF,constant KAPPA_REF,1.769400e-10,0.9487,diagnostic flattening of concentration dependence
2,transport_nonlinearity_amplified,default * (c_e/C_REF)^power,default * (c_e/C_REF)^power,1.769400e-10,0.9487,"diagnostic amplified concentration dependence,..."


In [7]:
# Cell 7 — Build final runnable branch registry（构建最终可运行分支注册表）

DAY37_RUNNABLE_BRANCH_SPECS = [
    {
        "branch_id": "baseline_default",
        "branch_type": "baseline",
        "surface_form": "false",
        "transport_branch_id": "baseline_default_transport",
        "transport_modification": "default",
        "physical_status": "baseline",
        "include_in_day37B": True,
        "interpretation_boundary": "Reference branch.",
    },
    {
        "branch_id": "double_layer_differential",
        "branch_type": "double_layer",
        "surface_form": "differential",
        "transport_branch_id": "baseline_default_transport",
        "transport_modification": "default",
        "physical_status": "pybamm_supported",
        "include_in_day37B": True,
        "interpretation_boundary": "Day36-supported differential surface-form branch.",
    },
    {
        "branch_id": "transport_flattened",
        "branch_type": "transport_parameterization",
        "surface_form": "false",
        "transport_branch_id": "transport_flattened",
        "transport_modification": "flattened",
        "physical_status": "diagnostic_not_truth_model",
        "include_in_day37B": True,
        "interpretation_boundary": "One-factor diagnostic branch; removes concentration dependence around reference values.",
    },
    {
        "branch_id": "transport_nonlinearity_amplified",
        "branch_type": "transport_parameterization",
        "surface_form": "false",
        "transport_branch_id": "transport_nonlinearity_amplified",
        "transport_modification": "nonlinearity_amplified",
        "physical_status": "diagnostic_stress_test_not_truth_model",
        "include_in_day37B": True,
        "interpretation_boundary": "One-factor diagnostic branch; amplifies concentration dependence.",
    },
    {
        "branch_id": "combined_sensitivity",
        "branch_type": "combined",
        "surface_form": "differential",
        "transport_branch_id": "transport_nonlinearity_amplified",
        "transport_modification": "nonlinearity_amplified",
        "physical_status": "diagnostic_stress_test_not_truth_model",
        "include_in_day37B": True,
        "interpretation_boundary": "Combined diagnostic branch: differential surface form + amplified transport nonlinearity.",
    },
    {
        "branch_id": "ocp_hysteresis_dynamic",
        "branch_type": "ocp_hysteresis",
        "surface_form": "false",
        "transport_branch_id": "baseline_default_transport",
        "transport_modification": "default",
        "physical_status": "deferred",
        "include_in_day37B": False,
        "interpretation_boundary": "Deferred: no supported PyBaMM option/model branch identified in Day37A.",
    },
    {
        "branch_id": "sei_solvent_diffusion_limited",
        "branch_type": "side_reaction",
        "surface_form": "false",
        "transport_branch_id": "baseline_default_transport",
        "transport_modification": "default",
        "physical_status": "pybamm_supported_but_excluded_from_clean_matrix",
        "include_in_day37B": False,
        "interpretation_boundary": "Builds, but it is a degradation side-reaction branch and would confound clean reversible non-geometric acceleration audit.",
    },
]

day37_branch_registry_final_df = pd.DataFrame(DAY37_RUNNABLE_BRANCH_SPECS)

out = DATA_DIR / "day37_branch_registry_final.csv"
day37_branch_registry_final_df.to_csv(out, index=False)

print("[OK] saved final Day37 branch registry:", out)
display(day37_branch_registry_final_df)

[OK] saved final Day37 branch registry: /Users/louislu/pybamm-dcac-superimposed/data/day37_branch_registry_final.csv


,branch_id,branch_type,surface_form,transport_branch_id,transport_modification,physical_status,include_in_day37B,interpretation_boundary
0,baseline_default,baseline,false,baseline_default_transport,default,baseline,True,Reference branch.
1,double_layer_differential,double_layer,differential,baseline_default_transport,default,pybamm_supported,True,Day36-supported differential surface-form branch.
2,transport_flattened,transport_parameterization,false,transport_flattened,flattened,diagnostic_not_truth_model,True,One-factor diagnostic branch; removes concentr...
3,transport_nonlinearity_amplified,transport_parameterization,false,transport_nonlinearity_amplified,nonlinearity_amplified,diagnostic_stress_test_not_truth_model,True,One-factor diagnostic branch; amplifies concen...
4,combined_sensitivity,combined,differential,transport_nonlinearity_amplified,nonlinearity_amplified,diagnostic_stress_test_not_truth_model,True,Combined diagnostic branch: differential surfa...
5,ocp_hysteresis_dynamic,ocp_hysteresis,false,baseline_default_transport,default,deferred,False,Deferred: no supported PyBaMM option/model bra...
6,sei_solvent_diffusion_limited,side_reaction,false,baseline_default_transport,default,pybamm_supported_but_excluded_from_clean_matrix,False,"Builds, but it is a degradation side-reaction ..."


In [8]:
# Cell 8 — Branch parameter values and build smoke test（分支参数与构建冒烟测试）

def make_parameter_values_for_day37_branch(branch_spec):
    pv = pybamm.ParameterValues(PRIMARY_PARAMETER_SET)
    transport_branch_id = branch_spec["transport_branch_id"]

    if transport_branch_id == "baseline_default_transport":
        return pv

    if transport_branch_id == "transport_flattened":
        pv.update(
            {
                "Electrolyte diffusivity [m2.s-1]": electrolyte_diffusivity_flattened,
                "Electrolyte conductivity [S.m-1]": electrolyte_conductivity_flattened,
            },
            check_already_exists=False,
        )
        return pv

    if transport_branch_id == "transport_nonlinearity_amplified":
        pv.update(
            {
                "Electrolyte diffusivity [m2.s-1]": electrolyte_diffusivity_nonlinearity_amplified,
                "Electrolyte conductivity [S.m-1]": electrolyte_conductivity_nonlinearity_amplified,
            },
            check_already_exists=False,
        )
        return pv

    raise ValueError(f"Unknown transport_branch_id: {transport_branch_id}")

def make_day37_model(branch_spec):
    return pybamm.lithium_ion.DFN(
        {
            "thermal": "isothermal",
            "surface form": branch_spec["surface_form"],
        }
    )

REQUIRED_KEYS_DAY37 = [
    "Negative electrode surface potential difference [V]",
    "Electrolyte concentration [mol.m-3]",
    "X-averaged negative electrode reaction overpotential [V]",
    "Average negative particle stoichiometry",
    "X-averaged negative particle surface stoichiometry",
]

build_rows = []

for _, branch in day37_branch_registry_final_df.iterrows():
    if not branch["include_in_day37B"]:
        build_rows.append({
            "branch_id": branch["branch_id"],
            "include_in_day37B": False,
            "build_status": "not_included",
            "n_variables": np.nan,
            "missing_required_keys": "",
            "error": "",
        })
        continue

    print("[BUILD]", branch["branch_id"])

    try:
        model = make_day37_model(branch)
        pv = make_parameter_values_for_day37_branch(branch)
        sim = pybamm.Simulation(model, parameter_values=pv)
        sim.build()
        keys = sorted(list(sim.built_model.variables.keys()))
        missing = [k for k in REQUIRED_KEYS_DAY37 if k not in keys]

        build_rows.append({
            "branch_id": branch["branch_id"],
            "include_in_day37B": True,
            "build_status": "ok",
            "n_variables": len(keys),
            "missing_required_keys": "; ".join(missing),
            "error": "",
        })

        print("[OK]", branch["branch_id"], "n_variables:", len(keys))

    except Exception as e:
        build_rows.append({
            "branch_id": branch["branch_id"],
            "include_in_day37B": True,
            "build_status": "failed",
            "n_variables": np.nan,
            "missing_required_keys": "",
            "error": repr(e),
        })
        print("[FAILED]", branch["branch_id"], repr(e))

day37_branch_build_audit_df = pd.DataFrame(build_rows)

out = DATA_DIR / "day37_branch_build_audit.csv"
day37_branch_build_audit_df.to_csv(out, index=False)

print("[OK] saved branch build audit:", out)
display(day37_branch_build_audit_df)

if (day37_branch_build_audit_df.query("include_in_day37B == True")["build_status"] != "ok").any():
    print("[WARN] Some included branches failed build. Inspect before running Day37B matrix.")
else:
    print("[OK] all included Day37B branches build successfully.")


[BUILD] baseline_default
[OK] baseline_default n_variables: 515
[BUILD] double_layer_differential
[OK] double_layer_differential n_variables: 522
[BUILD] transport_flattened
[OK] transport_flattened n_variables: 515
[BUILD] transport_nonlinearity_amplified
[OK] transport_nonlinearity_amplified n_variables: 515
[BUILD] combined_sensitivity
[OK] combined_sensitivity n_variables: 522
[OK] saved branch build audit: /Users/louislu/pybamm-dcac-superimposed/data/day37_branch_build_audit.csv


,branch_id,include_in_day37B,build_status,n_variables,missing_required_keys,error
0,baseline_default,True,ok,515.0,,
1,double_layer_differential,True,ok,522.0,,
2,transport_flattened,True,ok,515.0,,
3,transport_nonlinearity_amplified,True,ok,515.0,,
4,combined_sensitivity,True,ok,522.0,,
5,ocp_hysteresis_dynamic,False,not_included,NaN,,
6,sei_solvent_diffusion_limited,False,not_included,NaN,,


[OK] all included Day37B branches build successfully.


In [9]:
# Cell 9 — Final Day37B protocol matrix and waveform audit（最终Day37B协议矩阵与波形审计）

# Reference Q for scheduled policy.
candidate_ref_files = [
    DATA_DIR / "day35_run_summary.csv",
    DATA_DIR / "day34_run_summary.csv",
    DATA_DIR / "day33b_run_summary.csv",
]

Q_REF_AH = None
for path in candidate_ref_files:
    if path.exists():
        df = pd.read_csv(path)
        dc_rows = df[
            df["condition"].astype(str).str.contains("DC03")
            & ~df["condition"].astype(str).str.contains("AC")
        ]
        if len(dc_rows) > 0 and "Q_end_Ah" in dc_rows.columns:
            Q_REF_AH = float(dc_rows["Q_end_Ah"].iloc[0])
            print("[OK] Q_REF_AH loaded from:", path)
            break

if Q_REF_AH is None:
    Q_REF_AH = 4.4928790028035
    print("[WARN] fallback Q_REF_AH used:", Q_REF_AH)

Q_NOM_AH = float(parameter_values["Nominal cell capacity [A.h]"])
I_DC_A = DC_C * Q_NOM_AH

def qfrac_to_time_s(q_frac):
    return q_frac * Q_REF_AH * 3600.0 / I_DC_A

PROTOCOL_SPECS_DAY37B = [
    {
        "protocol_id": "DC03",
        "protocol_type": "DC",
        "case_role": "dc_reference",
        "schedule_type": "constant",
        "fixed_ac_c": 0.0,
        "early_ac_c": 0.0,
        "mid_ac_c": 0.0,
        "late_ac_c": 0.0,
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
    },
    {
        "protocol_id": "fixed_AC0p38",
        "protocol_type": "DCAC",
        "case_role": "near_boundary_baseline",
        "schedule_type": "constant",
        "fixed_ac_c": 0.38,
        "early_ac_c": 0.38,
        "mid_ac_c": 0.38,
        "late_ac_c": 0.38,
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
    },
    {
        "protocol_id": "fixed_AC0p7",
        "protocol_type": "DCAC",
        "case_role": "high_amplitude_stress_control",
        "schedule_type": "constant",
        "fixed_ac_c": 0.70,
        "early_ac_c": 0.70,
        "mid_ac_c": 0.70,
        "late_ac_c": 0.70,
        "qfrac_derate": np.nan,
        "qfrac_ac_off": np.nan,
    },
    {
        "protocol_id": "sched_v2_conservative",
        "protocol_type": "DCAC_SCHEDULED",
        "case_role": "day35_primary_candidate",
        "schedule_type": "qfrac_time_proxy",
        "fixed_ac_c": np.nan,
        "early_ac_c": 0.38,
        "mid_ac_c": 0.10,
        "late_ac_c": 0.00,
        "qfrac_derate": 0.70,
        "qfrac_ac_off": 0.80,
    },
]

protocol_rows = []

for _, branch in day37_branch_registry_final_df.query("include_in_day37B == True").iterrows():
    if day37_branch_build_audit_df.query("branch_id == @branch.branch_id")["build_status"].iloc[0] != "ok":
        continue

    for proto in PROTOCOL_SPECS_DAY37B:
        protocol_rows.append({
            "branch_id": branch["branch_id"],
            "branch_type": branch["branch_type"],
            "surface_form": branch["surface_form"],
            "transport_branch_id": branch["transport_branch_id"],
            "transport_modification": branch["transport_modification"],
            "physical_status": branch["physical_status"],
            "condition": f"Chen2020_DFN_Day37_{branch['branch_id']}_{proto['protocol_id']}",
            "parameter_set": PRIMARY_PARAMETER_SET,
            "model_type": MODEL_TYPE,
            "dc_c": DC_C,
            "tau_factor_set": TAU_FACTOR_MAIN,
            "tau_ref_set_s": TAU_REF_SET_S,
            "f_Hz": F_MAIN_HZ,
            "period_s": PERIOD_MAIN_S,
            **proto,
        })

day37_protocol_df = pd.DataFrame(protocol_rows)
day37_protocol_df["peak_c_nominal"] = day37_protocol_df["dc_c"] + day37_protocol_df[["fixed_ac_c", "early_ac_c"]].max(axis=1)

out = DATA_DIR / "day37_protocol_table.csv"
day37_protocol_df.to_csv(out, index=False)

print("[OK] saved final Day37 protocol table:", out)
display(day37_protocol_df)


def ac_amplitude_c_at_time_day37(t_s, protocol_row):
    stype = protocol_row["schedule_type"]
    t_s = np.asarray(t_s, dtype=float)

    if stype == "constant":
        return np.full_like(t_s, float(protocol_row["fixed_ac_c"]), dtype=float)

    if stype == "qfrac_time_proxy":
        t_derate = qfrac_to_time_s(float(protocol_row["qfrac_derate"]))
        t_off = qfrac_to_time_s(float(protocol_row["qfrac_ac_off"]))
        ac = np.zeros_like(t_s, dtype=float)
        ac[t_s < t_derate] = float(protocol_row["early_ac_c"])
        ac[(t_s >= t_derate) & (t_s < t_off)] = float(protocol_row["mid_ac_c"])
        ac[t_s >= t_off] = float(protocol_row["late_ac_c"])
        return ac

    raise ValueError(f"Unknown schedule_type: {stype}")

def charge_current_A_day37(t_s, protocol_row, q_nom_Ah=Q_NOM_AH):
    t_s = np.asarray(t_s, dtype=float)
    i_dc = float(protocol_row["dc_c"]) * q_nom_Ah
    ac_c = ac_amplitude_c_at_time_day37(t_s, protocol_row)
    i_ac = ac_c * q_nom_Ah

    if pd.isna(protocol_row["f_Hz"]) or float(protocol_row["f_Hz"]) == 0:
        return i_dc + np.zeros_like(t_s)

    return i_dc + i_ac * np.sin(2.0 * np.pi * float(protocol_row["f_Hz"]) * t_s)

wave_rows = []

for _, row in day37_protocol_df.iterrows():
    t_grid = np.arange(0.0, 12000.0 + 1.0, 1.0)
    i_charge = charge_current_A_day37(t_grid, row.to_dict())
    q_geom = cumulative_trapezoid_np(i_charge, t_grid) / 3600.0
    ac_c = ac_amplitude_c_at_time_day37(t_grid, row.to_dict())

    wave_rows.append({
        "branch_id": row["branch_id"],
        "protocol_id": row["protocol_id"],
        "condition": row["condition"],
        "I_charge_min_A": float(np.min(i_charge)),
        "I_charge_max_A": float(np.max(i_charge)),
        "Q_geom_end_Ah": float(np.max(q_geom)),
        "AC_C_min": float(np.min(ac_c)),
        "AC_C_max": float(np.max(ac_c)),
    })

day37_waveform_summary_df = pd.DataFrame(wave_rows)

out = DATA_DIR / "day37_current_waveform_summary.csv"
day37_waveform_summary_df.to_csv(out, index=False)

print("[OK] saved Day37 waveform summary:", out)
display(day37_waveform_summary_df)


[OK] Q_REF_AH loaded from: /Users/louislu/pybamm-dcac-superimposed/data/day35_run_summary.csv
[OK] saved final Day37 protocol table: /Users/louislu/pybamm-dcac-superimposed/data/day37_protocol_table.csv


,branch_id,branch_type,surface_form,transport_branch_id,transport_modification,physical_status,condition,parameter_set,model_type,dc_c,...,protocol_type,case_role,schedule_type,fixed_ac_c,early_ac_c,mid_ac_c,late_ac_c,qfrac_derate,qfrac_ac_off,peak_c_nominal
0,baseline_default,baseline,false,baseline_default_transport,default,baseline,Chen2020_DFN_Day37_baseline_default_DC03,Chen2020,DFN,0.3,...,DC,dc_reference,constant,0.00,0.00,0.00,0.00,NaN,NaN,0.30
1,baseline_default,baseline,false,baseline_default_transport,default,baseline,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,Chen2020,DFN,0.3,...,DCAC,near_boundary_baseline,constant,0.38,0.38,0.38,0.38,NaN,NaN,0.68
2,baseline_default,baseline,false,baseline_default_transport,default,baseline,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,Chen2020,DFN,0.3,...,DCAC,high_amplitude_stress_control,constant,0.70,0.70,0.70,0.70,NaN,NaN,1.00
3,baseline_default,baseline,false,baseline_default_transport,default,baseline,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,Chen2020,DFN,0.3,...,DCAC_SCHEDULED,day35_primary_candidate,qfrac_time_proxy,NaN,0.38,0.10,0.00,0.7,0.8,0.68
4,double_layer_differential,double_layer,differential,baseline_default_transport,default,pybamm_supported,Chen2020_DFN_Day37_double_layer_differential_DC03,Chen2020,DFN,0.3,...,DC,dc_reference,constant,0.00,0.00,0.00,0.00,NaN,NaN,0.30
5,double_layer_differential,double_layer,differential,baseline_default_transport,default,pybamm_supported,Chen2020_DFN_Day37_double_layer_differential_f...,Chen2020,DFN,0.3,...,DCAC,near_boundary_baseline,constant,0.38,0.38,0.38,0.38,NaN,NaN,0.68
6,double_layer_differential,double_layer,differential,baseline_default_transport,default,pybamm_supported,Chen2020_DFN_Day37_double_layer_differential_f...,Chen2020,DFN,0.3,...,DCAC,high_amplitude_stress_control,constant,0.70,0.70,0.70,0.70,NaN,NaN,1.00
7,double_layer_differential,double_layer,differential,baseline_default_transport,default,pybamm_supported,Chen2020_DFN_Day37_double_layer_differential_s...,Chen2020,DFN,0.3,...,DCAC_SCHEDULED,day35_primary_candidate,qfrac_time_proxy,NaN,0.38,0.10,0.00,0.7,0.8,0.68
8,transport_flattened,transport_parameterization,false,transport_flattened,flattened,diagnostic_not_truth_model,Chen2020_DFN_Day37_transport_flattened_DC03,Chen2020,DFN,0.3,...,DC,dc_reference,constant,0.00,0.00,0.00,0.00,NaN,NaN,0.30
9,transport_flattened,transport_parameterization,false,transport_flattened,flattened,diagnostic_not_truth_model,Chen2020_DFN_Day37_transport_flattened_fixed_A...,Chen2020,DFN,0.3,...,DCAC,near_boundary_baseline,constant,0.38,0.38,0.38,0.38,NaN,NaN,0.68


NameError: name 'cumulative_trapezoid_np' is not defined

In [10]:
# Cell 9B — Patch cumulative integration and rerun waveform audit（修正积分函数并重跑波形审计）

def cumulative_trapezoid_np(y, x):
    y = np.asarray(y, dtype=float).reshape(-1)
    x = np.asarray(x, dtype=float).reshape(-1)

    if len(y) != len(x):
        raise ValueError("x and y length mismatch")

    if len(y) < 2:
        return np.zeros_like(y)

    dx = np.diff(x)
    area = 0.5 * (y[:-1] + y[1:]) * dx

    return np.concatenate([[0.0], np.cumsum(area)])


wave_rows = []

for _, row in day37_protocol_df.iterrows():
    t_grid = np.arange(0.0, 12000.0 + 1.0, 1.0)
    i_charge = charge_current_A_day37(t_grid, row.to_dict())
    q_geom = cumulative_trapezoid_np(i_charge, t_grid) / 3600.0
    ac_c = ac_amplitude_c_at_time_day37(t_grid, row.to_dict())

    wave_rows.append({
        "branch_id": row["branch_id"],
        "branch_type": row["branch_type"],
        "transport_branch_id": row["transport_branch_id"],
        "transport_modification": row["transport_modification"],
        "protocol_id": row["protocol_id"],
        "condition": row["condition"],
        "I_charge_min_A": float(np.min(i_charge)),
        "I_charge_max_A": float(np.max(i_charge)),
        "Q_geom_end_Ah": float(np.max(q_geom)),
        "AC_C_min": float(np.min(ac_c)),
        "AC_C_max": float(np.max(ac_c)),
    })

day37_waveform_summary_df = pd.DataFrame(wave_rows)

out = DATA_DIR / "day37_current_waveform_summary.csv"
day37_waveform_summary_df.to_csv(out, index=False)

print("[OK] saved Day37 waveform summary:", out)
display(day37_waveform_summary_df)

[OK] saved Day37 waveform summary: /Users/louislu/pybamm-dcac-superimposed/data/day37_current_waveform_summary.csv


,branch_id,branch_type,transport_branch_id,transport_modification,protocol_id,condition,I_charge_min_A,I_charge_max_A,Q_geom_end_Ah,AC_C_min,AC_C_max
0,baseline_default,baseline,baseline_default_transport,default,DC03,Chen2020_DFN_Day37_baseline_default_DC03,1.5,1.5,5.000000,0.00,0.00
1,baseline_default,baseline,baseline_default_transport,default,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,-0.4,3.4,5.001986,0.38,0.38
2,baseline_default,baseline,baseline_default_transport,default,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,-2.0,5.0,5.031069,0.70,0.70
3,baseline_default,baseline,baseline_default_transport,default,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,-0.4,3.4,5.006584,0.00,0.38
4,double_layer_differential,double_layer,baseline_default_transport,default,DC03,Chen2020_DFN_Day37_double_layer_differential_DC03,1.5,1.5,5.000000,0.00,0.00
5,double_layer_differential,double_layer,baseline_default_transport,default,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,-0.4,3.4,5.001986,0.38,0.38
6,double_layer_differential,double_layer,baseline_default_transport,default,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,-2.0,5.0,5.031069,0.70,0.70
7,double_layer_differential,double_layer,baseline_default_transport,default,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,-0.4,3.4,5.006584,0.00,0.38
8,transport_flattened,transport_parameterization,transport_flattened,flattened,DC03,Chen2020_DFN_Day37_transport_flattened_DC03,1.5,1.5,5.000000,0.00,0.00
9,transport_flattened,transport_parameterization,transport_flattened,flattened,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,-0.4,3.4,5.001986,0.38,0.38


## Day37B2 — Full mechanism / parameterization sensitivity matrix

The Day37B runnable matrix is now frozen:

Branches:

1. `baseline_default`
2. `double_layer_differential`
3. `transport_flattened`
4. `transport_nonlinearity_amplified`
5. `combined_sensitivity`

Protocols:

- `DC03`
- `fixed_AC0p38`
- `fixed_AC0p7`
- `sched_v2_conservative`

`fixed_AC0p7` is retained as the mandatory high-amplitude stress control.

Day37B2 now runs the full matrix and evaluates:

- terminal-Q timing gain,
- geometry–residual decomposition,
- negative-electrode plating margin,
- microstate stress context,
- equal-terminal-Q internal-state differences,
- branch-level deltas relative to `baseline_default`.

This is still a diagnostic matrix, not a truth model.

In [11]:
# Cell 10 — Day37 full-matrix simulation utilities（Day37完整矩阵仿真工具）

def make_parameter_values_with_day37_current(protocol_row, max_time_s=24000, current_dt_s=1.0):
    """
    Build branch-specific parameter values and impose interpolated prescribed current.

    PyBaMM sign convention:
    +I = discharge
    -I = charge
    """
    branch = day37_branch_registry_final_df[
        day37_branch_registry_final_df["branch_id"] == protocol_row["branch_id"]
    ].iloc[0]

    pv = make_parameter_values_for_day37_branch(branch)
    q_nom_Ah = float(pv["Nominal cell capacity [A.h]"])

    t_grid = np.arange(0.0, max_time_s + current_dt_s, current_dt_s)
    i_charge_grid = charge_current_A_day37(t_grid, protocol_row, q_nom_Ah=q_nom_Ah)
    i_py_grid = -i_charge_grid

    current_interpolant = pybamm.Interpolant(
        t_grid,
        i_py_grid,
        pybamm.t,
        interpolator="linear",
    )

    pv.update({"Current function [A]": current_interpolant}, check_already_exists=False)

    q_geom_grid = cumulative_trapezoid_np(i_charge_grid, t_grid) / 3600.0
    ac_c_grid = ac_amplitude_c_at_time_day37(t_grid, protocol_row)

    geom_df = pd.DataFrame({
        "t_s": t_grid,
        "I_charge_A": i_charge_grid,
        "I_py_A": i_py_grid,
        "Q_geom_Ah": q_geom_grid,
        "AC_C_scheduled": ac_c_grid,
    })

    meta = {
        "nominal_capacity_Ah": q_nom_Ah,
        "max_time_s": max_time_s,
        "current_dt_s": current_dt_s,
        "phase_convention": "charge_first",
        "current_definition": "I_py(t) = -I_charge(t), interpolated prescribed current",
        "branch_id": protocol_row["branch_id"],
        "transport_branch_id": protocol_row["transport_branch_id"],
        "surface_form": protocol_row["surface_form"],
    }

    return pv, geom_df, meta


def extract_basic_timeseries(sol):
    t_s = np.asarray(sol.t, dtype=float)
    I_py_A = np.asarray(sol["Current [A]"](t_s), dtype=float).reshape(-1)
    V_terminal_V = np.asarray(sol["Terminal voltage [V]"](t_s), dtype=float).reshape(-1)

    I_charge_A = -I_py_A
    Q_net_Ah = cumulative_trapezoid_np(I_charge_A, t_s) / 3600.0

    return pd.DataFrame({
        "t_s": t_s,
        "I_py_A": I_py_A,
        "I_charge_A": I_charge_A,
        "V_terminal_V": V_terminal_V,
        "Q_net_Ah": Q_net_Ah,
    })


def run_day37_simulation(protocol_row, initial_soc=0.05, max_time_s=24000, n_eval=5000, current_dt_s=1.0):
    branch = day37_branch_registry_final_df[
        day37_branch_registry_final_df["branch_id"] == protocol_row["branch_id"]
    ].iloc[0]

    model = make_day37_model(branch)
    pv, geom_df, meta = make_parameter_values_with_day37_current(
        protocol_row=protocol_row,
        max_time_s=max_time_s,
        current_dt_s=current_dt_s,
    )

    sim = pybamm.Simulation(model, parameter_values=pv)
    t_eval = np.linspace(0.0, max_time_s, n_eval)
    sol = sim.solve(t_eval=t_eval, initial_soc=0.05)
    ts = extract_basic_timeseries(sol)

    return sim, sol, ts, geom_df, meta

print("[OK] Day37 full-matrix simulation utilities ready")

[OK] Day37 full-matrix simulation utilities ready


In [12]:
# Cell 11 — Run Day37 full sensitivity matrix（运行Day37完整敏感性矩阵）

solutions = {}
run_rows = []

for _, row in day37_protocol_df.iterrows():
    cond = row["condition"]
    print(f"\n[RUN] {cond}")

    try:
        sim_i, sol_i, ts_i, geom_i, meta_i = run_day37_simulation(row.to_dict())

        solutions[cond] = {
            "sim": sim_i,
            "sol": sol_i,
            "ts": ts_i,
            "geom": geom_i,
            "meta": meta_i,
            "protocol": row.to_dict(),
        }

        run_rows.append({
            "branch_id": row["branch_id"],
            "branch_type": row["branch_type"],
            "transport_branch_id": row["transport_branch_id"],
            "transport_modification": row["transport_modification"],
            "surface_form": row["surface_form"],
            "protocol_id": row["protocol_id"],
            "condition": cond,
            "case_role": row["case_role"],
            "status": "ok",
            "termination": str(sol_i.termination),
            "t_end_s": float(ts_i["t_s"].iloc[-1]),
            "t_end_min": float(ts_i["t_s"].iloc[-1] / 60.0),
            "Q_end_Ah": float(ts_i["Q_net_Ah"].iloc[-1]),
            "V_end_V": float(ts_i["V_terminal_V"].iloc[-1]),
            "I_charge_min_A": float(ts_i["I_charge_A"].min()),
            "I_charge_max_A": float(ts_i["I_charge_A"].max()),
            "error": "",
        })

        print("[OK]", cond, "| termination:", sol_i.termination)

    except Exception as e:
        run_rows.append({
            "branch_id": row["branch_id"],
            "branch_type": row["branch_type"],
            "transport_branch_id": row["transport_branch_id"],
            "transport_modification": row["transport_modification"],
            "surface_form": row["surface_form"],
            "protocol_id": row["protocol_id"],
            "condition": cond,
            "case_role": row["case_role"],
            "status": "failed",
            "termination": "",
            "t_end_s": np.nan,
            "t_end_min": np.nan,
            "Q_end_Ah": np.nan,
            "V_end_V": np.nan,
            "I_charge_min_A": np.nan,
            "I_charge_max_A": np.nan,
            "error": repr(e),
        })
        print("[FAILED]", cond, repr(e))

day37_run_summary_df = pd.DataFrame(run_rows)

out = DATA_DIR / "day37_run_summary.csv"
day37_run_summary_df.to_csv(out, index=False)

print("[OK] saved Day37 run summary:", out)
display(day37_run_summary_df)

if (day37_run_summary_df["status"] != "ok").any():
    print("[WARN] Some simulations failed. Inspect before continuing.")
else:
    print("[OK] all Day37 simulations completed.")



[RUN] Chen2020_DFN_Day37_baseline_default_DC03
[OK] Chen2020_DFN_Day37_baseline_default_DC03 | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_baseline_default_fixed_AC0p38
[OK] Chen2020_DFN_Day37_baseline_default_fixed_AC0p38 | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_baseline_default_fixed_AC0p7
[OK] Chen2020_DFN_Day37_baseline_default_fixed_AC0p7 | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_baseline_default_sched_v2_conservative
[OK] Chen2020_DFN_Day37_baseline_default_sched_v2_conservative | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_double_layer_differential_DC03
[OK] Chen2020_DFN_Day37_double_layer_differential_DC03 | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_double_layer_differential_fixed_AC0p38
[OK] Chen2020_DFN_Day37_double_layer_differential_fixed_AC0p38 | termination: event: Maximum voltage [V]

[RUN] Chen2020_DFN_Day37_double_layer_differential_fixed_AC0p7
[OK]

,branch_id,branch_type,transport_branch_id,transport_modification,surface_form,protocol_id,condition,case_role,status,termination,t_end_s,t_end_min,Q_end_Ah,V_end_V,I_charge_min_A,I_charge_max_A,error
0,baseline_default,baseline,baseline_default_transport,default,false,DC03,Chen2020_DFN_Day37_baseline_default_DC03,dc_reference,ok,event: Maximum voltage [V],10782.909607,179.715160,4.492879,4.2,1.500000,1.500000,
1,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,near_boundary_baseline,ok,event: Maximum voltage [V],9318.002845,155.300047,3.906509,4.2,-0.399987,3.399991,
2,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,high_amplitude_stress_control,ok,event: Maximum voltage [V],8627.927468,143.798791,3.633945,4.2,-1.999972,4.999984,
3,baseline_default,baseline,baseline_default_transport,default,false,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,day35_primary_candidate,ok,event: Maximum voltage [V],10767.715806,179.461930,4.493120,4.2,-0.399987,3.399991,
4,double_layer_differential,double_layer,baseline_default_transport,default,differential,DC03,Chen2020_DFN_Day37_double_layer_differential_DC03,dc_reference,ok,event: Maximum voltage [V],10782.315056,179.705251,4.492631,4.2,1.500000,1.500000,
5,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,near_boundary_baseline,ok,event: Maximum voltage [V],9317.735022,155.295584,3.906268,4.2,-0.399992,3.399994,
6,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,high_amplitude_stress_control,ok,event: Maximum voltage [V],8627.980703,143.799678,3.634027,4.2,-1.999977,4.999975,
7,double_layer_differential,double_layer,baseline_default_transport,default,differential,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,day35_primary_candidate,ok,event: Maximum voltage [V],10767.749746,179.462496,4.493142,4.2,-0.399992,3.399994,
8,transport_flattened,transport_parameterization,transport_flattened,flattened,false,DC03,Chen2020_DFN_Day37_transport_flattened_DC03,dc_reference,ok,event: Maximum voltage [V],10778.691290,179.644855,4.491121,4.2,1.500000,1.500000,
9,transport_flattened,transport_parameterization,transport_flattened,flattened,false,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,near_boundary_baseline,ok,event: Maximum voltage [V],9316.742656,155.279044,3.905328,4.2,-0.399989,3.399993,


[OK] all Day37 simulations completed.


In [13]:
# Cell 12 — Extract Day37 state, margin, and microstate variables（提取Day37状态、余量与微观变量）

NE_SURFACE_KEY = "Negative electrode surface potential difference [V]"
NE_SEPARATOR_KEY = "Negative electrode surface potential difference at separator interface [V]"
NE_XAVG_KEY = "X-averaged negative electrode surface potential difference [V]"
ELYTE_C_KEY = "Electrolyte concentration [mol.m-3]"
ETA_N_KEY = "X-averaged negative electrode reaction overpotential [V]"
ETA_P_KEY = "X-averaged positive electrode reaction overpotential [V]"
NEG_AVG_STO_KEY = "Average negative particle stoichiometry"
NEG_SURF_STO_XAVG_KEY = "X-averaged negative particle surface stoichiometry"
NEG_SURF_STO_MAX_KEY = "Maximum negative particle surface stoichiometry"
NEG_SURF_STO_MIN_KEY = "Minimum negative particle surface stoichiometry"
POS_AVG_STO_KEY = "Average positive particle stoichiometry"

def reduce_to_timeseries(arr, t_s, reducer="mean"):
    arr = np.asarray(arr, dtype=float)
    t_s = np.asarray(t_s, dtype=float)
    n_t = len(t_s)

    if arr.ndim == 0:
        return np.full(n_t, float(arr))
    if arr.ndim == 1:
        if arr.shape[0] == n_t:
            return arr
        return np.full(n_t, np.nanmean(arr))

    if arr.shape[-1] == n_t:
        flat = arr.reshape(-1, n_t)
        axis = 0
    elif arr.shape[0] == n_t:
        flat = arr.reshape(n_t, -1)
        axis = 1
    else:
        raise ValueError(f"Cannot identify time axis: arr.shape={arr.shape}, n_t={n_t}")

    if reducer == "min":
        return np.nanmin(flat, axis=axis)
    if reducer == "max":
        return np.nanmax(flat, axis=axis)
    if reducer == "mean":
        return np.nanmean(flat, axis=axis)
    raise ValueError(reducer)


def safe_extract(sol, key, t_s, reducer="mean"):
    try:
        return reduce_to_timeseries(sol[key](t_s), t_s, reducer=reducer)
    except Exception:
        return np.full(len(t_s), np.nan)


def classify_margin(u_v):
    if pd.isna(u_v):
        return "unresolved"
    if u_v <= 0.0:
        return "risk_flag"
    if u_v <= PLATING_MARGIN_SAFE_BUFFER_V:
        return "near_boundary"
    return "safe_margin"


state_ts_map = {}
extract_rows = []

for cond, obj in solutions.items():
    print("[EXTRACT]", cond)

    sol = obj["sol"]
    ts = obj["ts"].copy()
    t_s = ts["t_s"].to_numpy()

    # Plating margin proxy.
    ne_surface = sol[NE_SURFACE_KEY](t_s)
    ts["U_NE_min_V"] = reduce_to_timeseries(ne_surface, t_s, reducer="min")
    ts["U_NE_mean_V"] = reduce_to_timeseries(ne_surface, t_s, reducer="mean")
    ts["U_NE_separator_V"] = safe_extract(sol, NE_SEPARATOR_KEY, t_s, reducer="mean")
    ts["U_NE_xavg_V"] = safe_extract(sol, NE_XAVG_KEY, t_s, reducer="mean")
    ts["U_NE_min_mV"] = ts["U_NE_min_V"] * 1000.0

    # Transport state.
    c_e = sol[ELYTE_C_KEY](t_s)
    ts["c_e_min_molm3"] = reduce_to_timeseries(c_e, t_s, reducer="min")
    ts["c_e_max_molm3"] = reduce_to_timeseries(c_e, t_s, reducer="max")
    ts["c_e_mean_molm3"] = reduce_to_timeseries(c_e, t_s, reducer="mean")
    ts["c_e_range_molm3"] = ts["c_e_max_molm3"] - ts["c_e_min_molm3"]

    # State / kinetics.
    ts["eta_n_V"] = safe_extract(sol, ETA_N_KEY, t_s, reducer="mean")
    ts["eta_p_V"] = safe_extract(sol, ETA_P_KEY, t_s, reducer="mean")
    ts["neg_avg_sto"] = safe_extract(sol, NEG_AVG_STO_KEY, t_s, reducer="mean")
    ts["neg_surface_sto_xavg"] = safe_extract(sol, NEG_SURF_STO_XAVG_KEY, t_s, reducer="mean")
    ts["neg_surface_sto_max"] = safe_extract(sol, NEG_SURF_STO_MAX_KEY, t_s, reducer="mean")
    ts["neg_surface_sto_min"] = safe_extract(sol, NEG_SURF_STO_MIN_KEY, t_s, reducer="mean")
    ts["neg_surface_sto_range"] = ts["neg_surface_sto_max"] - ts["neg_surface_sto_min"]
    ts["pos_avg_sto"] = safe_extract(sol, POS_AVG_STO_KEY, t_s, reducer="mean")

    state_ts_map[cond] = ts

    extract_rows.append({
        "branch_id": obj["protocol"]["branch_id"],
        "branch_type": obj["protocol"]["branch_type"],
        "transport_branch_id": obj["protocol"]["transport_branch_id"],
        "transport_modification": obj["protocol"]["transport_modification"],
        "surface_form": obj["protocol"]["surface_form"],
        "protocol_id": obj["protocol"]["protocol_id"],
        "condition": cond,
        "n_time": len(ts),
        "min_U_NE_mV": float(ts["U_NE_min_mV"].min()),
        "fraction_time_below_50mV": float((ts["U_NE_min_V"] <= 0.050).mean()),
        "fraction_time_below_0mV": float((ts["U_NE_min_V"] <= 0.0).mean()),
        "max_c_e_range_molm3": float(ts["c_e_range_molm3"].max()),
        "max_neg_surface_sto_range": float(np.nanmax(ts["neg_surface_sto_range"])),
        "status": "ok",
    })

day37_state_extraction_audit_df = pd.DataFrame(extract_rows)

out = DATA_DIR / "day37_state_extraction_audit.csv"
day37_state_extraction_audit_df.to_csv(out, index=False)

print("[OK] saved Day37 state extraction audit:", out)
display(day37_state_extraction_audit_df)

[EXTRACT] Chen2020_DFN_Day37_baseline_default_DC03
[EXTRACT] Chen2020_DFN_Day37_baseline_default_fixed_AC0p38
[EXTRACT] Chen2020_DFN_Day37_baseline_default_fixed_AC0p7
[EXTRACT] Chen2020_DFN_Day37_baseline_default_sched_v2_conservative
[EXTRACT] Chen2020_DFN_Day37_double_layer_differential_DC03
[EXTRACT] Chen2020_DFN_Day37_double_layer_differential_fixed_AC0p38
[EXTRACT] Chen2020_DFN_Day37_double_layer_differential_fixed_AC0p7
[EXTRACT] Chen2020_DFN_Day37_double_layer_differential_sched_v2_conservative
[EXTRACT] Chen2020_DFN_Day37_transport_flattened_DC03
[EXTRACT] Chen2020_DFN_Day37_transport_flattened_fixed_AC0p38
[EXTRACT] Chen2020_DFN_Day37_transport_flattened_fixed_AC0p7
[EXTRACT] Chen2020_DFN_Day37_transport_flattened_sched_v2_conservative
[EXTRACT] Chen2020_DFN_Day37_transport_nonlinearity_amplified_DC03
[EXTRACT] Chen2020_DFN_Day37_transport_nonlinearity_amplified_fixed_AC0p38
[EXTRACT] Chen2020_DFN_Day37_transport_nonlinearity_amplified_fixed_AC0p7
[EXTRACT] Chen2020_DFN_Day37

,branch_id,branch_type,transport_branch_id,transport_modification,surface_form,protocol_id,condition,n_time,min_U_NE_mV,fraction_time_below_50mV,fraction_time_below_0mV,max_c_e_range_molm3,max_neg_surface_sto_range,status
0,baseline_default,baseline,baseline_default_transport,default,false,DC03,Chen2020_DFN_Day37_baseline_default_DC03,12159,37.280562,0.219837,0.000000,377.868423,0.116717,ok
1,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,14711,0.565686,0.143702,0.000000,747.684843,0.133279,ok
2,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,15500,-25.308203,0.220194,0.039484,1054.089062,0.173020,ok
3,baseline_default,baseline,baseline_default_transport,default,false,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,15575,24.160262,0.225169,0.000000,747.684843,0.133279,ok
4,double_layer_differential,double_layer,baseline_default_transport,default,differential,DC03,Chen2020_DFN_Day37_double_layer_differential_DC03,12263,37.287613,0.217973,0.000000,378.068256,0.116709,ok
5,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,13795,0.638102,0.147590,0.000000,748.395114,0.133295,ok
6,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,14577,-25.265352,0.228236,0.040681,1054.730226,0.173099,ok
7,double_layer_differential,double_layer,baseline_default_transport,default,differential,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,14810,24.180692,0.235044,0.000000,748.395114,0.133295,ok
8,transport_flattened,transport_parameterization,transport_flattened,flattened,false,DC03,Chen2020_DFN_Day37_transport_flattened_DC03,12156,36.498521,0.222442,0.000000,392.044690,0.126872,ok
9,transport_flattened,transport_parameterization,transport_flattened,flattened,false,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,14776,-1.151953,0.158094,0.002639,772.722593,0.147490,ok


In [14]:
# Cell 13 — Protocol-level margin and microstate summary（协议级余量与微观状态摘要）

protocol_rows = []

for _, pr in day37_protocol_df.iterrows():
    cond = pr["condition"]
    if cond not in state_ts_map:
        continue

    ts = state_ts_map[cond]
    idx_min = int(ts["U_NE_min_V"].idxmin())
    r = ts.loc[idx_min]

    protocol_rows.append({
        "branch_id": pr["branch_id"],
        "branch_type": pr["branch_type"],
        "transport_branch_id": pr["transport_branch_id"],
        "transport_modification": pr["transport_modification"],
        "surface_form": pr["surface_form"],
        "protocol_id": pr["protocol_id"],
        "condition": cond,
        "case_role": pr["case_role"],
        "min_U_NE_mV": float(r["U_NE_min_mV"]),
        "margin_class": classify_margin(float(r["U_NE_min_V"])),
        "fraction_time_below_50mV": float((ts["U_NE_min_V"] <= 0.050).mean()),
        "fraction_time_below_0mV": float((ts["U_NE_min_V"] <= 0.0).mean()),
        "Q_end_Ah": float(ts["Q_net_Ah"].max()),
        "Q_at_min_U_NE_Ah": float(r["Q_net_Ah"]),
        "t_at_min_U_NE_s": float(r["t_s"]),
        "I_charge_at_min_U_NE_A": float(r["I_charge_A"]),
        "I_charge_peak_observed_A": float(ts["I_charge_A"].max()),
        "V_terminal_at_min_U_NE_V": float(r["V_terminal_V"]),
        "distance_to_4p2V_at_min_U_NE_mV": float((4.2 - r["V_terminal_V"]) * 1000.0),
        "c_e_range_at_min_U_NE_molm3": float(r["c_e_range_molm3"]),
        "max_c_e_range_molm3": float(ts["c_e_range_molm3"].max()),
        "neg_avg_sto_at_min_U_NE": float(r["neg_avg_sto"]),
        "neg_surface_sto_xavg_at_min_U_NE": float(r["neg_surface_sto_xavg"]),
        "neg_surface_sto_range_at_min_U_NE": float(r["neg_surface_sto_range"]),
        "eta_n_at_min_U_NE_V": float(r["eta_n_V"]),
    })

day37_protocol_margin_microstate_summary_df = pd.DataFrame(protocol_rows)

out = DATA_DIR / "day37_protocol_margin_microstate_summary.csv"
day37_protocol_margin_microstate_summary_df.to_csv(out, index=False)

print("[OK] saved protocol margin/microstate summary:", out)
display(day37_protocol_margin_microstate_summary_df)


[OK] saved protocol margin/microstate summary: /Users/louislu/pybamm-dcac-superimposed/data/day37_protocol_margin_microstate_summary.csv


,branch_id,branch_type,transport_branch_id,transport_modification,surface_form,protocol_id,condition,case_role,min_U_NE_mV,margin_class,...,I_charge_at_min_U_NE_A,I_charge_peak_observed_A,V_terminal_at_min_U_NE_V,distance_to_4p2V_at_min_U_NE_mV,c_e_range_at_min_U_NE_molm3,max_c_e_range_molm3,neg_avg_sto_at_min_U_NE,neg_surface_sto_xavg_at_min_U_NE,neg_surface_sto_range_at_min_U_NE,eta_n_at_min_U_NE_V
0,baseline_default,baseline,baseline_default_transport,default,false,DC03,Chen2020_DFN_Day37_baseline_default_DC03,dc_reference,37.280562,near_boundary,...,1.500000,1.500000,4.200000,0.000000e+00,354.118549,377.868423,0.841523,0.846473,0.116717,-0.045310
1,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,near_boundary_baseline,0.565686,near_boundary,...,3.374140,3.399991,4.200000,-4.973799e-11,623.732719,747.684843,0.740809,0.749991,0.126834,-0.073212
2,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,high_amplitude_stress_control,-25.308203,risk_flag,...,4.953367,4.999984,4.194776,5.223883e+00,961.394009,1054.089062,0.675387,0.689397,0.133266,-0.091417
3,baseline_default,baseline,baseline_default_transport,default,false,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,day35_primary_candidate,24.160262,near_boundary,...,3.359285,3.399991,4.043495,1.565052e+02,733.219024,747.684843,0.597250,0.607167,0.094088,-0.068915
4,double_layer_differential,double_layer,baseline_default_transport,default,differential,DC03,Chen2020_DFN_Day37_double_layer_differential_DC03,dc_reference,37.287613,near_boundary,...,1.500000,1.500000,4.200000,0.000000e+00,354.196187,378.068256,0.841465,0.846415,0.116709,-0.045304
5,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,near_boundary_baseline,0.638102,near_boundary,...,3.372521,3.399994,4.200000,-4.973799e-11,622.540593,748.395114,0.740723,0.749890,0.126808,-0.073169
6,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,high_amplitude_stress_control,-25.265352,risk_flag,...,4.955663,4.999975,4.194435,5.564872e+00,960.806818,1054.730226,0.675008,0.689005,0.132863,-0.091410
7,double_layer_differential,double_layer,baseline_default_transport,default,differential,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,day35_primary_candidate,24.180692,near_boundary,...,3.360060,3.399994,4.043587,1.564128e+02,733.527023,748.395114,0.597175,0.607091,0.094101,-0.068920
8,transport_flattened,transport_parameterization,transport_flattened,flattened,false,DC03,Chen2020_DFN_Day37_transport_flattened_DC03,dc_reference,36.498521,near_boundary,...,1.500000,1.500000,4.200000,-8.881784e-13,362.791754,392.044690,0.841221,0.846171,0.126872,-0.045523
9,transport_flattened,transport_parameterization,transport_flattened,flattened,false,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,near_boundary_baseline,-1.151953,risk_flag,...,3.378614,3.399993,4.188859,1.114122e+01,688.604542,772.722593,0.719173,0.729021,0.135569,-0.072676


In [15]:
# Cell 14 — Equal-Q timing and geometry-residual matrix（等Q时间与几何-残差矩阵）

def first_passage_time(t_s, q_Ah, q_target_Ah):
    t_s = np.asarray(t_s, dtype=float)
    q_Ah = np.asarray(q_Ah, dtype=float)

    hit = np.where(q_Ah >= q_target_Ah)[0]
    if len(hit) == 0:
        return np.nan

    i = int(hit[0])
    if i == 0:
        return float(t_s[0])

    q0, q1 = q_Ah[i - 1], q_Ah[i]
    t0, t1 = t_s[i - 1], t_s[i]

    if q1 == q0:
        return float(t1)

    frac = (q_target_Ah - q0) / (q1 - q0)
    return float(t0 + frac * (t1 - t0))


def first_passage_from_geom_df(geom_df, q_target_Ah):
    return first_passage_time(
        geom_df["t_s"].to_numpy(),
        geom_df["Q_geom_Ah"].to_numpy(),
        q_target_Ah,
    )


eq_rows = []

for branch_id, g in day37_protocol_df.groupby("branch_id"):
    dc_cond = g[g["protocol_id"] == "DC03"]["condition"].iloc[0]

    if dc_cond not in state_ts_map:
        continue

    dc_ts = state_ts_map[dc_cond]
    dc_geom = solutions[dc_cond]["geom"]

    q_end_all = min(float(state_ts_map[c]["Q_net_Ah"].max()) for c in g["condition"] if c in state_ts_map)
    q_grid = np.linspace(0.2, 0.95 * q_end_all, 100)

    for _, pr in g.iterrows():
        cond = pr["condition"]
        if pr["protocol_id"] == "DC03" or cond not in state_ts_map:
            continue

        ts = state_ts_map[cond]
        geom = solutions[cond]["geom"]

        for q in q_grid:
            t_dc = first_passage_time(dc_ts["t_s"], dc_ts["Q_net_Ah"], q)
            t_protocol = first_passage_time(ts["t_s"], ts["Q_net_Ah"], q)

            t_dc_geom = first_passage_from_geom_df(dc_geom, q)
            t_protocol_geom = first_passage_from_geom_df(geom, q)

            eq_rows.append({
                "branch_id": pr["branch_id"],
                "branch_type": pr["branch_type"],
                "transport_branch_id": pr["transport_branch_id"],
                "transport_modification": pr["transport_modification"],
                "surface_form": pr["surface_form"],
                "protocol_id": pr["protocol_id"],
                "condition": cond,
                "Q_Ah": float(q),
                "delta_t_raw_s": t_dc - t_protocol,
                "delta_t_geom_s": t_dc_geom - t_protocol_geom,
                "delta_t_resid_s": (t_dc - t_protocol) - (t_dc_geom - t_protocol_geom),
            })

day37_equalq_df = pd.DataFrame(eq_rows)

out = DATA_DIR / "day37_equalQ_timing_geom_resid_curves.csv"
day37_equalq_df.to_csv(out, index=False)

day37_timing_summary_df = (
    day37_equalq_df
    .groupby(
        ["branch_id", "branch_type", "transport_branch_id", "transport_modification",
         "surface_form", "protocol_id", "condition"],
        as_index=False,
    )
    .agg(
        delta_t_raw_mean_s=("delta_t_raw_s", "mean"),
        delta_t_raw_min_s=("delta_t_raw_s", "min"),
        delta_t_raw_max_s=("delta_t_raw_s", "max"),
        delta_t_geom_mean_s=("delta_t_geom_s", "mean"),
        delta_t_resid_mean_s=("delta_t_resid_s", "mean"),
        resid_abs_mean_s=("delta_t_resid_s", lambda x: np.nanmean(np.abs(x))),
        resid_max_abs_s=("delta_t_resid_s", lambda x: np.nanmax(np.abs(x))),
        resid_positive_fraction=("delta_t_resid_s", lambda x: np.nanmean(np.asarray(x) > 0.0)),
    )
)

def classify_resid(row):
    raw = abs(row["delta_t_raw_mean_s"])
    if raw < 1:
        return "near_zero_or_unresolved"

    ratio = row["resid_abs_mean_s"] / raw

    if (row["delta_t_resid_mean_s"] > 5.0) and (row["resid_positive_fraction"] >= 0.75):
        return "candidate_positive_residual"

    if ratio < 0.05:
        return "geometry_dominated"
    if ratio < 0.20:
        return "mostly_geometry_small_residual"
    return "residual_non_negligible"

day37_timing_summary_df["dt_resid_class"] = day37_timing_summary_df.apply(classify_resid, axis=1)

out = DATA_DIR / "day37_equalQ_timing_geom_resid_summary.csv"
day37_timing_summary_df.to_csv(out, index=False)

print("[OK] saved Day37 equal-Q curves:", out)
print("[OK] saved Day37 timing summary:", DATA_DIR / "day37_equalQ_timing_geom_resid_summary.csv")
display(day37_timing_summary_df)


[OK] saved Day37 equal-Q curves: /Users/louislu/pybamm-dcac-superimposed/data/day37_equalQ_timing_geom_resid_summary.csv
[OK] saved Day37 timing summary: /Users/louislu/pybamm-dcac-superimposed/data/day37_equalQ_timing_geom_resid_summary.csv


,branch_id,branch_type,transport_branch_id,transport_modification,surface_form,protocol_id,condition,delta_t_raw_mean_s,delta_t_raw_min_s,delta_t_raw_max_s,delta_t_geom_mean_s,delta_t_resid_mean_s,resid_abs_mean_s,resid_max_abs_s,resid_positive_fraction,dt_resid_class
0,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,69.350655,-0.012173,137.953460,69.363536,-0.012881,0.012948,0.045133,0.01,geometry_dominated
1,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,158.665349,48.438917,254.159325,158.685568,-0.020219,0.020219,0.070742,0.00,geometry_dominated
2,baseline_default,baseline,baseline_default_transport,default,false,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,65.689082,-0.012173,137.953460,65.702825,-0.013742,0.013810,0.045133,0.01,geometry_dominated
3,combined_sensitivity,combined,transport_nonlinearity_amplified,nonlinearity_amplified,differential,fixed_AC0p38,Chen2020_DFN_Day37_combined_sensitivity_fixed_...,69.649727,0.002748,137.971237,69.659066,-0.009338,0.009455,0.158643,0.03,geometry_dominated
4,combined_sensitivity,combined,transport_nonlinearity_amplified,nonlinearity_amplified,differential,fixed_AC0p7,Chen2020_DFN_Day37_combined_sensitivity_fixed_...,159.477780,50.410360,254.096086,159.496201,-0.018422,0.018422,0.095960,0.00,geometry_dominated
5,combined_sensitivity,combined,transport_nonlinearity_amplified,nonlinearity_amplified,differential,sched_v2_conservative,Chen2020_DFN_Day37_combined_sensitivity_sched_...,67.161330,0.002748,137.971237,67.171040,-0.009710,0.009827,0.158643,0.03,geometry_dominated
6,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,69.391331,-0.002150,137.969001,69.398991,-0.007660,0.007775,0.033239,0.04,geometry_dominated
7,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,158.727476,48.563539,254.165386,158.743254,-0.015778,0.015778,0.044732,0.00,geometry_dominated
8,double_layer_differential,double_layer,baseline_default_transport,default,differential,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,65.723993,-0.002150,137.969001,65.731800,-0.007807,0.007922,0.033239,0.04,geometry_dominated
9,transport_flattened,transport_parameterization,transport_flattened,flattened,false,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,69.003987,-0.004513,137.961014,69.017278,-0.013291,0.013373,0.045909,0.01,geometry_dominated


In [16]:
# Cell 15 — Branch deltas versus baseline_default（相对baseline_default的分支差异）

# Merge timing + margin/microstate summaries.
day37_matrix_ledger_df = (
    day37_protocol_margin_microstate_summary_df
    .merge(
        day37_timing_summary_df[
            [
                "branch_id",
                "protocol_id",
                "delta_t_raw_mean_s",
                "delta_t_geom_mean_s",
                "delta_t_resid_mean_s",
                "resid_abs_mean_s",
                "resid_max_abs_s",
                "resid_positive_fraction",
                "dt_resid_class",
            ]
        ],
        on=["branch_id", "protocol_id"],
        how="left",
    )
)

# Fill DC reference timing values.
for col in ["delta_t_raw_mean_s", "delta_t_geom_mean_s", "delta_t_resid_mean_s", "resid_abs_mean_s", "resid_max_abs_s", "resid_positive_fraction"]:
    day37_matrix_ledger_df[col] = day37_matrix_ledger_df[col].fillna(0.0)
day37_matrix_ledger_df["dt_resid_class"] = day37_matrix_ledger_df["dt_resid_class"].fillna("dc_reference")

out = DATA_DIR / "day37_matrix_ledger.csv"
day37_matrix_ledger_df.to_csv(out, index=False)

print("[OK] saved Day37 matrix ledger:", out)
display(day37_matrix_ledger_df)


# Branch deltas relative to baseline_default for each protocol_id.
baseline_df = day37_matrix_ledger_df[
    day37_matrix_ledger_df["branch_id"] == "baseline_default"
].set_index("protocol_id")

delta_rows = []

for _, row in day37_matrix_ledger_df.iterrows():
    pid = row["protocol_id"]
    if pid not in baseline_df.index:
        continue

    base = baseline_df.loc[pid]

    delta_rows.append({
        "branch_id": row["branch_id"],
        "branch_type": row["branch_type"],
        "transport_modification": row["transport_modification"],
        "surface_form": row["surface_form"],
        "protocol_id": pid,
        "case_role": row["case_role"],

        # Absolute values
        "delta_t_raw_mean_s": row["delta_t_raw_mean_s"],
        "delta_t_resid_mean_s": row["delta_t_resid_mean_s"],
        "resid_abs_mean_s": row["resid_abs_mean_s"],
        "dt_resid_class": row["dt_resid_class"],
        "min_U_NE_mV": row["min_U_NE_mV"],
        "margin_class": row["margin_class"],
        "fraction_time_below_0mV": row["fraction_time_below_0mV"],
        "max_c_e_range_molm3": row["max_c_e_range_molm3"],
        "neg_surface_sto_range_at_min_U_NE": row["neg_surface_sto_range_at_min_U_NE"],
        "eta_n_at_min_U_NE_V": row["eta_n_at_min_U_NE_V"],

        # Deltas vs baseline_default for same protocol
        "delta_vs_baseline_raw_mean_s": row["delta_t_raw_mean_s"] - base["delta_t_raw_mean_s"],
        "delta_vs_baseline_resid_mean_s": row["delta_t_resid_mean_s"] - base["delta_t_resid_mean_s"],
        "delta_vs_baseline_resid_abs_mean_s": row["resid_abs_mean_s"] - base["resid_abs_mean_s"],
        "delta_vs_baseline_min_U_NE_mV": row["min_U_NE_mV"] - base["min_U_NE_mV"],
        "delta_vs_baseline_fraction_below_0mV": row["fraction_time_below_0mV"] - base["fraction_time_below_0mV"],
        "delta_vs_baseline_max_c_e_range_molm3": row["max_c_e_range_molm3"] - base["max_c_e_range_molm3"],
        "delta_vs_baseline_surface_range_at_min": row["neg_surface_sto_range_at_min_U_NE"] - base["neg_surface_sto_range_at_min_U_NE"],
        "delta_vs_baseline_eta_n_at_min_V": row["eta_n_at_min_U_NE_V"] - base["eta_n_at_min_U_NE_V"],
    })

day37_branch_delta_vs_baseline_df = pd.DataFrame(delta_rows)

out = DATA_DIR / "day37_branch_delta_vs_baseline.csv"
day37_branch_delta_vs_baseline_df.to_csv(out, index=False)

print("[OK] saved branch delta vs baseline:", out)
display(day37_branch_delta_vs_baseline_df)

[OK] saved Day37 matrix ledger: /Users/louislu/pybamm-dcac-superimposed/data/day37_matrix_ledger.csv


,branch_id,branch_type,transport_branch_id,transport_modification,surface_form,protocol_id,condition,case_role,min_U_NE_mV,margin_class,...,neg_surface_sto_xavg_at_min_U_NE,neg_surface_sto_range_at_min_U_NE,eta_n_at_min_U_NE_V,delta_t_raw_mean_s,delta_t_geom_mean_s,delta_t_resid_mean_s,resid_abs_mean_s,resid_max_abs_s,resid_positive_fraction,dt_resid_class
0,baseline_default,baseline,baseline_default_transport,default,false,DC03,Chen2020_DFN_Day37_baseline_default_DC03,dc_reference,37.280562,near_boundary,...,0.846473,0.116717,-0.045310,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference
1,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p38,Chen2020_DFN_Day37_baseline_default_fixed_AC0p38,near_boundary_baseline,0.565686,near_boundary,...,0.749991,0.126834,-0.073212,69.350655,69.363536,-0.012881,0.012948,0.045133,0.01,geometry_dominated
2,baseline_default,baseline,baseline_default_transport,default,false,fixed_AC0p7,Chen2020_DFN_Day37_baseline_default_fixed_AC0p7,high_amplitude_stress_control,-25.308203,risk_flag,...,0.689397,0.133266,-0.091417,158.665349,158.685568,-0.020219,0.020219,0.070742,0.00,geometry_dominated
3,baseline_default,baseline,baseline_default_transport,default,false,sched_v2_conservative,Chen2020_DFN_Day37_baseline_default_sched_v2_c...,day35_primary_candidate,24.160262,near_boundary,...,0.607167,0.094088,-0.068915,65.689082,65.702825,-0.013742,0.013810,0.045133,0.01,geometry_dominated
4,double_layer_differential,double_layer,baseline_default_transport,default,differential,DC03,Chen2020_DFN_Day37_double_layer_differential_DC03,dc_reference,37.287613,near_boundary,...,0.846415,0.116709,-0.045304,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference
5,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p38,Chen2020_DFN_Day37_double_layer_differential_f...,near_boundary_baseline,0.638102,near_boundary,...,0.749890,0.126808,-0.073169,69.391331,69.398991,-0.007660,0.007775,0.033239,0.04,geometry_dominated
6,double_layer_differential,double_layer,baseline_default_transport,default,differential,fixed_AC0p7,Chen2020_DFN_Day37_double_layer_differential_f...,high_amplitude_stress_control,-25.265352,risk_flag,...,0.689005,0.132863,-0.091410,158.727476,158.743254,-0.015778,0.015778,0.044732,0.00,geometry_dominated
7,double_layer_differential,double_layer,baseline_default_transport,default,differential,sched_v2_conservative,Chen2020_DFN_Day37_double_layer_differential_s...,day35_primary_candidate,24.180692,near_boundary,...,0.607091,0.094101,-0.068920,65.723993,65.731800,-0.007807,0.007922,0.033239,0.04,geometry_dominated
8,transport_flattened,transport_parameterization,transport_flattened,flattened,false,DC03,Chen2020_DFN_Day37_transport_flattened_DC03,dc_reference,36.498521,near_boundary,...,0.846171,0.126872,-0.045523,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference
9,transport_flattened,transport_parameterization,transport_flattened,flattened,false,fixed_AC0p38,Chen2020_DFN_Day37_transport_flattened_fixed_A...,near_boundary_baseline,-1.151953,risk_flag,...,0.729021,0.135569,-0.072676,69.003987,69.017278,-0.013291,0.013373,0.045909,0.01,geometry_dominated


[OK] saved branch delta vs baseline: /Users/louislu/pybamm-dcac-superimposed/data/day37_branch_delta_vs_baseline.csv


,branch_id,branch_type,transport_modification,surface_form,protocol_id,case_role,delta_t_raw_mean_s,delta_t_resid_mean_s,resid_abs_mean_s,dt_resid_class,...,neg_surface_sto_range_at_min_U_NE,eta_n_at_min_U_NE_V,delta_vs_baseline_raw_mean_s,delta_vs_baseline_resid_mean_s,delta_vs_baseline_resid_abs_mean_s,delta_vs_baseline_min_U_NE_mV,delta_vs_baseline_fraction_below_0mV,delta_vs_baseline_max_c_e_range_molm3,delta_vs_baseline_surface_range_at_min,delta_vs_baseline_eta_n_at_min_V
0,baseline_default,baseline,default,false,DC03,dc_reference,0.000000,0.000000,0.000000,dc_reference,...,0.116717,-0.045310,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,baseline_default,baseline,default,false,fixed_AC0p38,near_boundary_baseline,69.350655,-0.012881,0.012948,geometry_dominated,...,0.126834,-0.073212,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,baseline_default,baseline,default,false,fixed_AC0p7,high_amplitude_stress_control,158.665349,-0.020219,0.020219,geometry_dominated,...,0.133266,-0.091417,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,baseline_default,baseline,default,false,sched_v2_conservative,day35_primary_candidate,65.689082,-0.013742,0.013810,geometry_dominated,...,0.094088,-0.068915,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,double_layer_differential,double_layer,default,differential,DC03,dc_reference,0.000000,0.000000,0.000000,dc_reference,...,0.116709,-0.045304,0.000000,0.000000,0.000000,0.007051,0.000000,0.199834,-0.000007,0.000007
5,double_layer_differential,double_layer,default,differential,fixed_AC0p38,near_boundary_baseline,69.391331,-0.007660,0.007775,geometry_dominated,...,0.126808,-0.073169,0.040676,0.005221,-0.005173,0.072416,0.000000,0.710270,-0.000026,0.000043
6,double_layer_differential,double_layer,default,differential,fixed_AC0p7,high_amplitude_stress_control,158.727476,-0.015778,0.015778,geometry_dominated,...,0.132863,-0.091410,0.062127,0.004441,-0.004441,0.042851,0.001197,0.641163,-0.000403,0.000006
7,double_layer_differential,double_layer,default,differential,sched_v2_conservative,day35_primary_candidate,65.723993,-0.007807,0.007922,geometry_dominated,...,0.094101,-0.068920,0.034910,0.005935,-0.005887,0.020429,0.000000,0.710270,0.000014,-0.000005
8,transport_flattened,transport_parameterization,flattened,false,DC03,dc_reference,0.000000,0.000000,0.000000,dc_reference,...,0.126872,-0.045523,0.000000,0.000000,0.000000,-0.782040,0.000000,14.176267,0.010155,-0.000213
9,transport_flattened,transport_parameterization,flattened,false,fixed_AC0p38,near_boundary_baseline,69.003987,-0.013291,0.013373,geometry_dominated,...,0.135569,-0.072676,-0.346668,-0.000410,0.000425,-1.717638,0.002639,25.037749,0.008735,0.000535


In [17]:
# Cell 16 — Non-geometric acceleration candidate screen（非几何加速候选筛查）

candidate_rows = []

for _, row in day37_matrix_ledger_df.iterrows():
    if row["protocol_id"] == "DC03":
        candidate_class = "dc_reference"
    elif row["dt_resid_class"] == "candidate_positive_residual":
        candidate_class = "candidate_positive_non_geometric"
    elif (row["delta_t_resid_mean_s"] > 1.0) and (row["resid_positive_fraction"] > 0.5):
        candidate_class = "weak_positive_residual_signal"
    elif abs(row["delta_t_resid_mean_s"]) <= 1.0:
        candidate_class = "no_material_residual_acceleration"
    elif row["delta_t_resid_mean_s"] < -1.0:
        candidate_class = "negative_residual_or_penalty"
    else:
        candidate_class = "unclassified"

    candidate_rows.append({
        "branch_id": row["branch_id"],
        "branch_type": row["branch_type"],
        "transport_modification": row["transport_modification"],
        "surface_form": row["surface_form"],
        "protocol_id": row["protocol_id"],
        "delta_t_raw_mean_s": row["delta_t_raw_mean_s"],
        "delta_t_geom_mean_s": row["delta_t_geom_mean_s"],
        "delta_t_resid_mean_s": row["delta_t_resid_mean_s"],
        "resid_abs_mean_s": row["resid_abs_mean_s"],
        "resid_max_abs_s": row["resid_max_abs_s"],
        "resid_positive_fraction": row["resid_positive_fraction"],
        "dt_resid_class": row["dt_resid_class"],
        "candidate_class": candidate_class,
        "min_U_NE_mV": row["min_U_NE_mV"],
        "margin_class": row["margin_class"],
        "max_c_e_range_molm3": row["max_c_e_range_molm3"],
    })

day37_non_geometric_candidate_screen_df = pd.DataFrame(candidate_rows)

out = DATA_DIR / "day37_non_geometric_candidate_screen.csv"
day37_non_geometric_candidate_screen_df.to_csv(out, index=False)

print("[OK] saved non-geometric candidate screen:", out)

display_cols = [
    "branch_id",
    "protocol_id",
    "delta_t_raw_mean_s",
    "delta_t_geom_mean_s",
    "delta_t_resid_mean_s",
    "resid_abs_mean_s",
    "resid_positive_fraction",
    "dt_resid_class",
    "candidate_class",
    "min_U_NE_mV",
    "margin_class",
]

display(day37_non_geometric_candidate_screen_df[display_cols])


# Summary counts.
candidate_count_df = (
    day37_non_geometric_candidate_screen_df
    .groupby(["candidate_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_cases"})
)

out = DATA_DIR / "day37_non_geometric_candidate_counts.csv"
candidate_count_df.to_csv(out, index=False)

print("[OK] saved candidate count summary:", out)
display(candidate_count_df)


[OK] saved non-geometric candidate screen: /Users/louislu/pybamm-dcac-superimposed/data/day37_non_geometric_candidate_screen.csv


,branch_id,protocol_id,delta_t_raw_mean_s,delta_t_geom_mean_s,delta_t_resid_mean_s,resid_abs_mean_s,resid_positive_fraction,dt_resid_class,candidate_class,min_U_NE_mV,margin_class
0,baseline_default,DC03,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference,dc_reference,37.280562,near_boundary
1,baseline_default,fixed_AC0p38,69.350655,69.363536,-0.012881,0.012948,0.01,geometry_dominated,no_material_residual_acceleration,0.565686,near_boundary
2,baseline_default,fixed_AC0p7,158.665349,158.685568,-0.020219,0.020219,0.00,geometry_dominated,no_material_residual_acceleration,-25.308203,risk_flag
3,baseline_default,sched_v2_conservative,65.689082,65.702825,-0.013742,0.013810,0.01,geometry_dominated,no_material_residual_acceleration,24.160262,near_boundary
4,double_layer_differential,DC03,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference,dc_reference,37.287613,near_boundary
5,double_layer_differential,fixed_AC0p38,69.391331,69.398991,-0.007660,0.007775,0.04,geometry_dominated,no_material_residual_acceleration,0.638102,near_boundary
6,double_layer_differential,fixed_AC0p7,158.727476,158.743254,-0.015778,0.015778,0.00,geometry_dominated,no_material_residual_acceleration,-25.265352,risk_flag
7,double_layer_differential,sched_v2_conservative,65.723993,65.731800,-0.007807,0.007922,0.04,geometry_dominated,no_material_residual_acceleration,24.180692,near_boundary
8,transport_flattened,DC03,0.000000,0.000000,0.000000,0.000000,0.00,dc_reference,dc_reference,36.498521,near_boundary
9,transport_flattened,fixed_AC0p38,69.003987,69.017278,-0.013291,0.013373,0.01,geometry_dominated,no_material_residual_acceleration,-1.151953,risk_flag


[OK] saved candidate count summary: /Users/louislu/pybamm-dcac-superimposed/data/day37_non_geometric_candidate_counts.csv


,candidate_class,n_cases
0,dc_reference,5
1,no_material_residual_acceleration,15


In [18]:
# Cell 17 — Branch-level sensitivity synthesis（分支级敏感性综合）

branch_synthesis_rows = []

for branch_id, g in day37_matrix_ledger_df.groupby("branch_id"):
    branch_meta = g.iloc[0]
    non_dc = g[g["protocol_id"] != "DC03"]

    n_candidate = int((non_dc["dt_resid_class"] == "candidate_positive_residual").sum())
    max_resid_mean = float(non_dc["delta_t_resid_mean_s"].max())
    min_resid_mean = float(non_dc["delta_t_resid_mean_s"].min())
    max_abs_resid_mean = float(non_dc["resid_abs_mean_s"].max())

    n_risk = int((g["margin_class"] == "risk_flag").sum())
    min_u = float(g["min_U_NE_mV"].min())
    max_ce = float(g["max_c_e_range_molm3"].max())

    if n_candidate > 0:
        residual_verdict = "positive_residual_candidate_present"
    elif max_resid_mean > 1.0:
        residual_verdict = "weak_positive_residual_possible"
    else:
        residual_verdict = "no_positive_non_geometric_acceleration"

    if n_risk > 0:
        margin_verdict = "contains_hard_margin_risk"
    else:
        margin_verdict = "no_hard_margin_violation"

    branch_synthesis_rows.append({
        "branch_id": branch_id,
        "branch_type": branch_meta["branch_type"],
        "transport_modification": branch_meta["transport_modification"],
        "surface_form": branch_meta["surface_form"],
        "n_protocols": len(g),
        "n_positive_residual_candidates": n_candidate,
        "max_delta_t_resid_mean_s": max_resid_mean,
        "min_delta_t_resid_mean_s": min_resid_mean,
        "max_resid_abs_mean_s": max_abs_resid_mean,
        "residual_verdict": residual_verdict,
        "n_margin_risk_cases": n_risk,
        "min_U_NE_mV_across_protocols": min_u,
        "max_c_e_range_molm3_across_protocols": max_ce,
        "margin_verdict": margin_verdict,
    })

day37_branch_synthesis_df = pd.DataFrame(branch_synthesis_rows)

out = DATA_DIR / "day37_branch_sensitivity_synthesis.csv"
day37_branch_synthesis_df.to_csv(out, index=False)

print("[OK] saved branch sensitivity synthesis:", out)
display(day37_branch_synthesis_df)

[OK] saved branch sensitivity synthesis: /Users/louislu/pybamm-dcac-superimposed/data/day37_branch_sensitivity_synthesis.csv


,branch_id,branch_type,transport_modification,surface_form,n_protocols,n_positive_residual_candidates,max_delta_t_resid_mean_s,min_delta_t_resid_mean_s,max_resid_abs_mean_s,residual_verdict,n_margin_risk_cases,min_U_NE_mV_across_protocols,max_c_e_range_molm3_across_protocols,margin_verdict
0,baseline_default,baseline,default,false,4,0,-0.012881,-0.020219,0.020219,no_positive_non_geometric_acceleration,1,-25.308203,1054.089062,contains_hard_margin_risk
1,combined_sensitivity,combined,nonlinearity_amplified,differential,4,0,-0.009338,-0.018422,0.018422,no_positive_non_geometric_acceleration,2,-34.150417,1158.689136,contains_hard_margin_risk
2,double_layer_differential,double_layer,default,differential,4,0,-0.007660,-0.015778,0.015778,no_positive_non_geometric_acceleration,1,-25.265352,1054.730226,contains_hard_margin_risk
3,transport_flattened,transport_parameterization,flattened,false,4,0,-0.013291,-0.021110,0.021110,no_positive_non_geometric_acceleration,2,-28.479797,1069.612800,contains_hard_margin_risk
4,transport_nonlinearity_amplified,transport_parameterization,nonlinearity_amplified,false,4,0,-0.013573,-0.022664,0.022664,no_positive_non_geometric_acceleration,2,-34.204651,1158.771074,contains_hard_margin_risk


In [19]:
# Cell 18 — Day37 final synthesis table（Day37最终综合表）

final_claim_rows = [
    {
        "claim": "Transport parameterization branches create stable positive non-geometric terminal-Q acceleration",
        "evidence": "Flattened, amplified, and combined transport branches remain geometry-dominated across fixed AC0.38, fixed AC0.7, and sched_v2.",
        "verdict": "not_supported",
        "boundary": "Diagnostic transport branches are not physical truth models; result applies to Chen2020/DFN/isothermal tested matrix.",
    },
    {
        "claim": "Double-layer differential branch creates non-geometric acceleration",
        "evidence": "Double-layer branch remains geometry-dominated and matches Day36 conclusion.",
        "verdict": "not_supported",
        "boundary": "Default Chen2020 C_dl and tested frequencies only.",
    },
    {
        "claim": "Transport branches affect plating-margin / microstate stress",
        "evidence": "Flattened and amplified transport branches shift min U_NE and increase electrolyte-gradient / surface-state stress, especially for fixed AC0.7.",
        "verdict": "supported",
        "boundary": "These are sensitivity diagnostics; not validated physical transport alternatives.",
    },
    {
        "claim": "fixed AC0.7 is necessary stress control",
        "evidence": "fixed AC0.7 remains the largest raw-gain and highest-risk protocol across branches; transport perturbations amplify hard-margin risk.",
        "verdict": "supported",
        "boundary": "Stress control, not admissible candidate.",
    },
    {
        "claim": "sched_v2 remains the preferred admissible candidate under sensitivity branches",
        "evidence": "sched_v2 remains above 0 mV in all runnable branches, but margin decreases under amplified/combined transport.",
        "verdict": "partially_supported",
        "boundary": "Still near-boundary; thermal/aging/cross-parameter transfer remain untested.",
    },
    {
        "claim": "OCP / hysteresis branch was audited",
        "evidence": "No supported current PyBaMM/Chen2020 DFN option was identified; branch remains deferred.",
        "verdict": "deferred",
        "boundary": "Requires a valid PyBaMM hysteresis/dynamic-OCP model branch or custom model development.",
    },
]

day37_final_claim_synthesis_df = pd.DataFrame(final_claim_rows)

out = DATA_DIR / "day37_final_claim_synthesis.csv"
day37_final_claim_synthesis_df.to_csv(out, index=False)

print("[OK] saved Day37 final claim synthesis:", out)
display(day37_final_claim_synthesis_df)

[OK] saved Day37 final claim synthesis: /Users/louislu/pybamm-dcac-superimposed/data/day37_final_claim_synthesis.csv


,claim,evidence,verdict,boundary
0,Transport parameterization branches create sta...,"Flattened, amplified, and combined transport b...",not_supported,Diagnostic transport branches are not physical...
1,Double-layer differential branch creates non-g...,Double-layer branch remains geometry-dominated...,not_supported,Default Chen2020 C_dl and tested frequencies o...
2,Transport branches affect plating-margin / mic...,Flattened and amplified transport branches shi...,supported,These are sensitivity diagnostics; not validat...
3,fixed AC0.7 is necessary stress control,fixed AC0.7 remains the largest raw-gain and h...,supported,"Stress control, not admissible candidate."
4,sched_v2 remains the preferred admissible cand...,sched_v2 remains above 0 mV in all runnable br...,partially_supported,Still near-boundary; thermal/aging/cross-param...
5,OCP / hysteresis branch was audited,No supported current PyBaMM/Chen2020 DFN optio...,deferred,Requires a valid PyBaMM hysteresis/dynamic-OCP...


## Final closure — Notebook 37

Notebook 37 built and executed a mechanism / parameterization sensitivity matrix before writing the non-geometric-acceleration claim review.

The goal was not to perform another blind parameter scan. The goal was to test whether plausible missing-physics or parameterization branches could create stable, positive, engineering-significant non-geometric terminal-Q acceleration.

The audited branches were:

- `baseline_default`
- `double_layer_differential`
- `transport_flattened`
- `transport_nonlinearity_amplified`
- `combined_sensitivity`

The deferred / excluded branches were:

- `ocp_hysteresis_dynamic`: deferred because no supported PyBaMM / Chen2020 DFN dynamic-OCP or hysteresis option was identified in the current audit.
- `sei_solvent_diffusion_limited`: excluded from the clean matrix because it is a degradation side-reaction branch and would confound reversible terminal-Q acceleration interpretation.
- particle mechanics / reversible plating options: failed capability audit under the current Chen2020 parameter set because required parameters were missing.

All runnable branches were tested with the same mandatory protocol set:

- DC `0.3C`
- fixed `0.3C + 0.38C`
- fixed `0.3C + 0.7C`
- scheduled `sched_v2_conservative`

The fixed `AC0.7` stress protocol was retained as the high-amplitude control.

The timing result was decisive. Across all runnable branches and all non-DC protocols, no case was classified as a positive non-geometric residual candidate. All non-DC protocols remained geometry-dominated:

- `Δt_raw_mean ≈ Δt_geom_mean`
- `Δt_resid_mean ≈ 0`
- candidate class: `no_material_residual_acceleration`

The candidate screen contained:

- `5` DC reference cases
- `15` non-DC cases with `no_material_residual_acceleration`
- `0` positive non-geometric acceleration candidates

Transport parameterization did affect the electrochemical and admissibility layers. Both `transport_flattened` and `transport_nonlinearity_amplified` shifted negative-electrode margin and increased electrolyte-gradient / surface-state stress. In particular, fixed `AC0.38` moved from near-boundary under the baseline branch to hard-margin risk under the transport diagnostic branches, and fixed `AC0.7` became more severe under transport perturbation.

Representative margin results:

- baseline fixed `AC0.38`: min U_NE ≈ +0.57 mV
- transport-flattened fixed `AC0.38`: min U_NE ≈ −1.15 mV
- transport-nonlinearity-amplified fixed `AC0.38`: min U_NE ≈ −4.10 mV
- baseline fixed `AC0.7`: min U_NE ≈ −25.31 mV
- transport-nonlinearity-amplified fixed `AC0.7`: min U_NE ≈ −34.20 mV

The scheduled `sched_v2_conservative` candidate remained above the 0 mV hard-margin proxy across all runnable branches, but its margin decreased under transport perturbations:

- baseline `sched_v2`: min U_NE ≈ +24.16 mV
- transport-flattened `sched_v2`: min U_NE ≈ +21.28 mV
- transport-nonlinearity-amplified `sched_v2`: min U_NE ≈ +16.74 mV
- combined-sensitivity `sched_v2`: min U_NE ≈ +16.76 mV

Correct conclusion wording:

> The Day37 mechanism / parameterization sensitivity matrix does not support stable, positive, engineering-significant non-geometric terminal-Q acceleration. Double-layer dynamics, flattened electrolyte transport, amplified electrolyte transport nonlinearity, and the combined sensitivity branch all remain geometry-dominated. These branches do affect plating-margin and microstate-stress severity, especially for fixed high-amplitude protocols, but they do not turn raw timing gain into non-geometric terminal-Q acceleration.

Interpretation boundary:

- Transport-flattened and transport-nonlinearity-amplified branches are diagnostic perturbations, not validated electrolyte models.
- `fixed_AC0.7` is a stress control, not an admissible candidate.
- `sched_v2_conservative` remains the preferred first-generation scheduled candidate, but it remains near-boundary and is not thermally or aging validated.
- OCP / hysteresis remains deferred because no supported model branch was identified in this audit.
- Day37 does not prove non-geometric acceleration is impossible in all real cells. It shows that it is not supported under the current Chen2020 / DFN / isothermal / prescribed-current / terminal-Q decomposition framework, even after the audited missing-physics and parameterization branches.

Next step:

The next document should be a formal claim review:

`docs/non_geometric_acceleration_claim_review_after_day37.md`

That review should separate:

1. supported claims,
2. unsupported claims,
3. deferred mechanisms,
4. metric-identifiability limits,
5. the recommended future direction: admissible DC–AC waveform scheduling rather than pursuit of terminal-Q non-geometric acceleration alone.
